# 00 - Original Exploration (Archival)

This notebook preserves the initial exploratory workflow. It is not part of the recommended reviewer path; use notebooks 01 through 04 for the cleaned, modular project narrative.



In [ ]:
# ====== Install + Imports ======
!pip -q install yfinance

import yfinance as yf
import pandas as pd
import numpy as np
from typing import List, Optional, Tuple, Union
import torch
import gc # Garbage Collector

# ====== Plotting imports ======
import os
from pathlib import Path
import matplotlib.pyplot as plt

# ====== GPU Utility Functions ======

def format_mem(size_bytes):
    """Helper to format bytes into MiB/GiB"""
    if size_bytes == 0:
        return "0 B"
    size_name = ("B", "KiB", "MiB", "GiB", "TiB")
    # Use int() for floor division, np.log for logarithm
    i = int(np.floor(np.log(size_bytes) / np.log(1024)))
    p = np.power(1024, i)
    s = round(size_bytes / p, 2)
    return f"{s} {size_name[i]}"

def print_gpu_stats(note=""):
    """Prints current GPU memory stats."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated()
        reserved = torch.cuda.memory_reserved()
        print(f"--- GPU Stats {note} ---")
        print(f"  Allocated: {format_mem(allocated)} (Memory used by tensors)")
        print(f"  Reserved:  {format_mem(reserved)} (Total memory held by PyTorch)")
        print("-------------------------")
    else:
        print("  > No GPU available.")

def clear_gpu_cache(print_stats=True):
    """
    Utility function to clear PyTorch's CUDA cache
    and run Python's garbage collection.
    Optionally prints stats before and after.
    """
    if torch.cuda.is_available():
        if print_stats:
            print_gpu_stats("(Before Clear)")

        print("  > Clearing GPU cache...")
        gc.collect()
        torch.cuda.empty_cache()
        print("  > GPU cache cleared.")

        if print_stats:
            print_gpu_stats("(After Clear)")
    else:
        print("  > No GPU available. Skipping cache clear.")

clear_gpu_cache()
print_gpu_stats("After clearing")

--- GPU Stats (Before Clear) ---
  Allocated: 0 B (Memory used by tensors)
  Reserved:  0 B (Total memory held by PyTorch)
-------------------------
  > Clearing GPU cache...
  > GPU cache cleared.
--- GPU Stats (After Clear) ---
  Allocated: 0 B (Memory used by tensors)
  Reserved:  0 B (Total memory held by PyTorch)
-------------------------
--- GPU Stats After clearing ---
  Allocated: 0 B (Memory used by tensors)
  Reserved:  0 B (Total memory held by PyTorch)
-------------------------


In [ ]:
# ====== Option-Chain Dataset Builder (Function + Caller) ======

def _time_to_maturity_years(expiry: Union[str, pd.Timestamp]) -> float:
    """ACT/365 time to maturity in years with robust tz handling."""
    now_aware = pd.Timestamp.utcnow()  # tz-aware
    exp_ts = pd.to_datetime(expiry)
    exp_naive = exp_ts.tz_convert(None) if exp_ts.tzinfo is not None else exp_ts.tz_localize(None)
    now_naive = now_aware.tz_convert(None)
    T = max((exp_naive - now_naive).total_seconds() / (365.25 * 24 * 3600), 0.0)
    return T

def _add_side_and_mid(df: pd.DataFrame, side: str) -> pd.DataFrame:
    """Add side ('C' or 'P') and compute robust mid (fallback to lastPrice)."""
    out = df.copy()
    out["side"] = side
    out["mid"] = np.where(
        out[["bid", "ask"]].notna().all(axis=1) & (out["ask"] > 0),
        (out["bid"] + out["ask"]) / 2.0,
        out["lastPrice"]
    )
    return out

def _robust_target_price(row: pd.Series) -> float:
    """Prefer mid clipped to [bid, ask]; else fall back to lastPrice."""
    b, a, m, lp = row.get("bid"), row.get("ask"), row.get("mid"), row.get("lastPrice")
    if pd.notna(b) and pd.notna(a) and a > 0:
        return min(max(m, b), a)
    return m if pd.notna(m) else lp

def build_option_dataset(
    ticker: str,
    expiries: Union[str, List[str], None] = "auto",
    tmin: float = 0.02,                 # ~≥ 1 week
    tmax: float = 0.75,                 # ≤ ~9 months
    moneyness_bounds: Tuple[float, float] = (0.6, 1.4),
    iv_bounds: Tuple[float, float] = (0.01, 3.0),   # 1%–300%
    require_open_interest: bool = True,
    save_csv: Optional[str] = None,
) -> Tuple[pd.DataFrame, float]:
    """
    Fetch + clean option chain(s) into a calibration-ready table.
    Returns (DataFrame, S0).
    """
    opt = yf.Ticker(ticker)

    # Pick expiries
    all_exps = list(opt.options or [])
    if isinstance(expiries, list):
        chosen_exps = [e for e in expiries if e in all_exps]
    elif expiries == "auto" or expiries is None:
        chosen_exps = [e for e in all_exps if tmin <= _time_to_maturity_years(e) <= tmax]
    else:
        chosen_exps = [expiries] if expiries in all_exps else []

    if not chosen_exps:
        raise ValueError(f"No expiries selected for {ticker}. Available: {all_exps[:5]}...")

    # Underlying spot (S0)
    hist = opt.history(period="5d", interval="1d")
    if len(hist) == 0:
        raise RuntimeError("No price history; check ticker/connection.")
    S0 = float(hist["Close"].iloc[-1])

    # Pull and assemble
    tables = []
    for ex in chosen_exps:
        ch = opt.option_chain(ex)
        calls = _add_side_and_mid(ch.calls, "C")
        puts  = _add_side_and_mid(ch.puts,  "P")
        t = pd.concat([calls, puts], ignore_index=True).rename(columns={"impliedVolatility": "iv"})
        t["iv"] = pd.to_numeric(t["iv"], errors="coerce")
        t["expiry"] = pd.to_datetime(ex)
        t["T"] = _time_to_maturity_years(ex)
        t["underlying"] = ticker
        tables.append(t)

    df = pd.concat(tables, ignore_index=True)

    # Features
    df["moneyness"] = df["strike"] / S0
    df["type"] = np.where(df["side"] == "C", 1, -1)
    df["iv_clamped"] = df["iv"].clip(lower=iv_bounds[0], upper=iv_bounds[1])

    # Liquidity + quality filters
    df = df[
        (df["mid"].notna()) &
        (df["mid"] > 0) &
        ((df["openInterest"].fillna(0) > 0) if require_open_interest else True) &
        (df["moneyness"].between(*moneyness_bounds))
    ].copy()

    # Target price for calibration
    df["target_price"] = df.apply(_robust_target_price, axis=1)

    # Final tidy columns
    cols = [
        "underlying", "expiry", "T", "side", "type", "strike", "moneyness",
        "bid", "ask", "mid", "lastPrice", "target_price",
        "openInterest", "volume", "iv", "iv_clamped", "inTheMoney", "currency"
    ]
    df = df[cols].sort_values(["expiry", "side", "strike"]).reset_index(drop=True)

    if save_csv:
        df.to_csv(save_csv, index=False)

    return df, S0

In [ ]:
# ====== Sanity / Validation Utilities ======
import math

def bs_price_from_iv(S, K, T, r, iv, side: str, q: float = 0.0) -> float:
    """
    Black–Scholes price using implied vol (no dividends by default).
    side: "C" or "P"
    """
    if T <= 0 or iv <= 0:
        return max(0.0, (S - K) if side == "C" else (K - S))
    if q != 0:
        # carry q via forward price F = S * exp((r - q) T)
        F = S * math.exp((r - q) * T)
        df = math.exp(-r * T)
        d1 = (math.log(F / K) + 0.5 * iv * iv * T) / (iv * math.sqrt(T))
        d2 = d1 - iv * math.sqrt(T)
        from math import erf, sqrt
        N = lambda x: 0.5 * (1.0 + erf(x / math.sqrt(2.0)))
        call = df * (F * N(d1) - K * N(d2))
    else:
        # standard form (q = 0)
        d1 = (math.log(S / K) + (r + 0.5 * iv * iv) * T) / (iv * math.sqrt(T))
        d2 = d1 - iv * math.sqrt(T)
        from math import erf, sqrt
        N = lambda x: 0.5 * (1.0 + erf(x / math.sqrt(2.0)))
        call = S * N(d1) - K * math.exp(-r * T) * N(d2)

    if side == "C":
        return call
    else:
        # put via parity
        return call - S * math.exp(-q * T) + K * math.exp(-r * T)

def put_call_parity_residual(S, K, T, r, c_price, p_price, q: float = 0.0) -> float:
    """
    Residual = C - P - (S*e^{-qT} - K*e^{-rT}) ; should be close to 0 if quotes consistent.
    """
    return c_price - p_price - (S * math.exp(-q * T) - K * math.exp(-r * T))

def validate_dataset(df, S0, r: float = 0.0, q: float = 0.0):
    """
    1) Reprice with BS from quoted IV (per row) and compare with target_price.
    2) Check put-call parity on matched strikes/maturities.
    Returns a summary dict and a per-row error column.
    """
    df = df.copy()
    # BS repro price from IV
    df["bs_repro"] = [
        bs_price_from_iv(S0, K, T, r, iv if iv > 0 else 0.0, side, q=q)
        for K, T, iv, side in zip(df["strike"], df["T"], df["iv_clamped"].fillna(0.0), df["side"])
    ]
    df["abs_err_vs_target"] = (df["bs_repro"] - df["target_price"]).abs()

    # Quick parity check on rows where we have both C and P at same K,T
    # Build pivot by (expiry, strike)
    small = df[["expiry","T","strike","side","target_price"]].copy()
    piv = small.pivot_table(index=["expiry","T","strike"], columns="side", values="target_price", aggfunc="median")
    piv = piv.dropna(subset=["C","P"], how="any").reset_index()
    piv["parity_residual"] = [
        put_call_parity_residual(S0, K, T, r, c, p, q=q)
        for K, T, c, p in zip(piv["strike"], piv["T"], piv["C"], piv["P"])
    ]

    summary = {
        "rows": len(df),
        "mean_abs_err_vs_target": float(df["abs_err_vs_target"].mean()) if len(df) else float("nan"),
        "median_abs_err_vs_target": float(df["abs_err_vs_target"].median()) if len(df) else float("nan"),
        "pct_large_err_(>1.0)": float((df["abs_err_vs_target"] > 1.0).mean() * 100.0) if len(df) else float("nan"),
        "parity_pairs": int(len(piv)),
        "parity_residual_mean": float(piv["parity_residual"].mean()) if len(piv) else float("nan"),
        "parity_residual_median": float(piv["parity_residual"].median()) if len(piv) else float("nan"),
    }
    return df, piv, summary

In [ ]:
# ====== Infer term structure (r, q) per expiry and re-check parity ======
from collections import defaultdict

def infer_r_q_per_expiry(df: pd.DataFrame, S0: float):
    """
    For each expiry: fit C - P ~ a + b*(-K)
      -> slope b = -DF, intercept a = DF*F
      -> DF = -b, F = a/DF
      -> r = -ln(DF)/T, q = r - ln(F/S0)/T
    Returns: dict[expiry] -> {"T", "DF", "F", "r", "q", "n_pairs"}
    """
    out = {}
    # build parity pairs by (expiry, T, strike)
    small = df[["expiry","T","strike","side","target_price"]].copy()
    piv = small.pivot_table(index=["expiry","T","strike"], columns="side", values="target_price", aggfunc="median").dropna(subset=["C","P"])
    if len(piv)==0:
        return out
    # group by expiry
    for (exp, T), block in piv.reset_index().groupby(["expiry","T"]):
        if T <= 0 or len(block) < 5:
            continue
        # regress y = C-P vs K: y ≈ A - DF*K  (no intercept trick: intercept = A = DF*F)
        K = block["strike"].values
        y = (block["C"] - block["P"]).values
        # fit y = alpha + beta*K
        X = np.column_stack([np.ones_like(K), K])
        alpha, beta = np.linalg.lstsq(X, y, rcond=None)[0]
        DF = max(1e-8, -beta)  # slope should be negative; guard tiny
        F  = alpha / DF if DF > 0 else np.nan
        r  = max(-0.5, -np.log(DF)/T) if DF>0 else np.nan               # crude bounds
        rq = (np.log(F/S0)/T) if (F>0 and S0>0) else np.nan
        q  = r - rq if np.isfinite(rq) else np.nan
        out[pd.to_datetime(exp)] = {"T": float(T), "DF": float(DF), "F": float(F), "r": float(r), "q": float(q), "n_pairs": int(len(block))}
    return out

def apply_r_q_and_check(df: pd.DataFrame, S0: float, rq_map: dict):
    df = df.copy()
    # attach r,q per expiry
    df["r"] = df["expiry"].map(lambda e: rq_map.get(pd.to_datetime(e), {}).get("r", 0.0))
    df["q"] = df["expiry"].map(lambda e: rq_map.get(pd.to_datetime(e), {}).get("q", 0.0))
    # recompute BS repricing using per-expiry r,q and iv
    def bs_price_from_iv_row(row):
        S, K, T, r, q, iv, side = S0, row["strike"], row["T"], row["r"], row["q"], row["iv_clamped"], row["side"]
        if T <= 0 or iv <= 0:
            return max(0.0, (S-K) if side=="C" else (K-S))
        d1 = (np.log(S/K) + (r - q + 0.5*iv*iv)*T) / (iv*np.sqrt(T))
        d2 = d1 - iv*np.sqrt(T)
        N  = lambda x: 0.5*(1.0 + math.erf(x/np.sqrt(2.0)))
        call = S*np.exp(-q*T)*N(d1) - K*np.exp(-r*T)*N(d2)
        return call if side=="C" else call - S*np.exp(-q*T) + K*np.exp(-r*T)

    df["bs_repro_rq"] = df.apply(bs_price_from_iv_row, axis=1)
    df["abs_err_vs_target_rq"] = (df["bs_repro_rq"] - df["target_price"]).abs()

    # parity residuals with per-expiry r,q
    small = df[["expiry","T","strike","side","target_price","r","q"]].copy()
    piv = small.pivot_table(index=["expiry","T","strike","r","q"], columns="side", values="target_price", aggfunc="median").dropna(subset=["C","P"]).reset_index()
    def parity_resid_row(row):
        S, K, T, r, q, c, p = S0, row["strike"], row["T"], row["r"], row["q"], row["C"], row["P"]
        return c - p - (S*np.exp(-q*T) - K*np.exp(-r*T))
    piv["parity_residual_rq"] = piv.apply(parity_resid_row, axis=1)

    summary = {
        "rows": int(len(df)),
        "abs_err_vs_target_rq_mean": float(df["abs_err_vs_target_rq"].mean()),
        "abs_err_vs_target_rq_median": float(df["abs_err_vs_target_rq"].median()),
        "parity_pairs": int(len(piv)),
        "parity_residual_rq_mean": float(piv["parity_residual_rq"].mean()) if len(piv) else float("nan"),
        "parity_residual_rq_median": float(piv["parity_residual_rq"].median()) if len(piv) else float("nan"),
    }
    return df, piv, summary

In [ ]:
# ====== Hi-quality plotting utils (no seaborn) ======
import os, math, json, time
from pathlib import Path
from typing import Iterable, Tuple
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Global, clean style (monospace labels are a nice quant touch)
mpl.rcParams.update({
    "figure.figsize": (8.5, 5.0),
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 12,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.linewidth": 0.6,
    "grid.alpha": 0.4,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

def _ensure_dir(path: str) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p

def _fname(base: str, kind: str) -> str:
    ts = time.strftime("%Y%m%d")
    return f"{base}_{ts}.{kind}"

def _save_all(fig: mpl.figure.Figure, outdir: Path, base: str):
    fig.tight_layout()
    for ext in ("png", "svg"):
        fig.savefig(outdir / _fname(base, ext), bbox_inches="tight")
    plt.close(fig)

# ---- Black–Scholes helpers (for bid/ask IV bands / sanity shading)
def _norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def _bs_call(S, K, T, r, q, vol):
    if T <= 0 or vol <= 0:
        return max(0.0, S*math.exp(-q*T) - K*math.exp(-r*T))
    sqrtT = math.sqrt(T)
    d1 = (math.log(S/K) + (r - q + 0.5*vol*vol)*T) / (vol*sqrtT)
    d2 = d1 - vol*sqrtT
    return S*math.exp(-q*T)*_norm_cdf(d1) - K*math.exp(-r*T)*_norm_cdf(d2)

def _bs_put(S, K, T, r, q, vol):
    c = _bs_call(S, K, T, r, q, vol)
    return c - S*math.exp(-q*T) + K*math.exp(-r*T)

def _iv_from_price(S, K, T, r, q, price, side, vol0=0.2):
    # Robust Newton + bisection fallback. Returns np.nan if impossible.
    if T <= 0 or price <= 0:
        return np.nan
    # bounds
    lo, hi = 1e-4, 5.0
    # quick check: if price > intrinsic upper bound, bail
    intrinsic = max(0.0, (S*math.exp(-q*T) - K*math.exp(-r*T)) if side=="C" else (K*math.exp(-r*T) - S*math.exp(-q*T)))
    theo_max = S*math.exp(-q*T) if side=="C" else K*math.exp(-r*T)
    if price < intrinsic - 1e-8 or price > theo_max + 1e-8:
        return np.nan

    v = max(lo, min(hi, vol0))
    for _ in range(20):
        c = _bs_call(S,K,T,r,q,v) if side=="C" else _bs_put(S,K,T,r,q,v)
        # vega
        if v <= 0: break
        d1 = (math.log(S/K) + (r - q + 0.5*v*v)*T) / (v*math.sqrt(T))
        vega = S*math.exp(-q*T)*math.sqrt(T) * (1.0/math.sqrt(2*math.pi)) * math.exp(-0.5*d1*d1)
        diff = c - price
        if abs(diff) < 1e-7: return v
        if vega > 1e-10:
            v = v - diff/vega
        # clamp
        if v < lo or v > hi:
            break
    # Bisection
    lo_v, hi_v = lo, hi
    for _ in range(60):
        mid = 0.5*(lo_v + hi_v)
        c_mid = _bs_call(S,K,T,r,q,mid) if side=="C" else _bs_put(S,K,T,r,q,mid)
        if c_mid > price: hi_v = mid
        else: lo_v = mid
        if hi_v - lo_v < 1e-5:
            return 0.5*(hi_v + lo_v)
    return np.nan

def _vol_band_from_quotes(df_row, S0, r=0.0, q=0.0):
    # Return (iv_bid, iv_mid, iv_ask) via inversion, where feasible.
    K = float(df_row["strike"]); T = float(df_row["T"]); side = df_row["side"]
    bid = float(df_row["bid"]) if pd.notna(df_row["bid"]) else np.nan
    ask = float(df_row["ask"]) if pd.notna(df_row["ask"]) else np.nan
    mid = float(df_row["mid"]) if pd.notna(df_row["mid"]) else np.nan
    iv_mid = _iv_from_price(S0, K, T, r, q, mid, side) if np.isfinite(mid) else np.nan
    iv_bid = _iv_from_price(S0, K, T, r, q, bid, side) if np.isfinite(bid) and bid>0 else np.nan
    iv_ask = _iv_from_price(S0, K, T, r, q, ask, side) if np.isfinite(ask) and ask>0 else np.nan
    return iv_bid, iv_mid, iv_ask

# Light smoothing (polyfit) for pretty curves without hiding structure
def _smooth_xy(x: np.ndarray, y: np.ndarray, deg: int = 3, n: int = 200) -> Tuple[np.ndarray, np.ndarray]:
    x, y = np.asarray(x), np.asarray(y)
    if len(x) < deg+1: return x, y
    order = np.argsort(x)
    x_s, y_s = x[order], y[order]
    # guard against duplicate x
    uniq, idx = np.unique(x_s, return_index=True)
    x_s, y_s = x_s[idx], y_s[idx]
    coeff = np.polyfit(x_s, y_s, deg)
    xx = np.linspace(x_s.min(), x_s.max(), n)
    yy = np.polyval(coeff, xx)
    return xx, yy

In [ ]:
# ====== Portfolio-grade diagnostics: save to ./figures_prelim ======
OUT = _ensure_dir("./figures_prelim")

def plot_iv_smile_with_bands(df, S0, expiry, r=0.0, q=0.0):
    dd = df[df["expiry"] == pd.to_datetime(expiry)].copy()
    if dd.empty: return
    fig = plt.figure()
    for side in ("C","P"):
        sub = dd[dd["side"]==side].sort_values("strike").copy()
        if sub.empty: continue
        # compute bid/mid/ask IV bands by inversion
        bands = np.array([_vol_band_from_quotes(row, S0, r, q) for _, row in sub.iterrows()], dtype=float)
        ivb, ivm, iva = bands[:,0], bands[:,1], bands[:,2]
        # plot mid IV (smoothed overlay) + bid/ask band as vertical errorbars
        plt.plot(sub["strike"].values, ivm, marker="o", linestyle="-", label=f"{side} (IV mid)")
        # vertical “whiskers”: bid–ask IV range
        for x, lo, hi in zip(sub["strike"].values, ivb, iva):
            if np.isfinite(lo) and np.isfinite(hi):
                plt.vlines(x, lo, hi, linewidth=1)
        # smooth overlay (polynomial fit)
        xx, yy = _smooth_xy(sub["strike"].values, ivm, deg=3)
        plt.plot(xx, yy, linestyle="--")
    # annotate ATM
    K_atm = S0
    plt.axvline(K_atm, linestyle="--")
    plt.title(f"Implied Vol Smile with Bid/Ask Bands — {pd.to_datetime(expiry).date()}")
    plt.xlabel("Strike"); plt.ylabel("Implied Volatility")
    plt.legend()
    _save_all(fig, OUT, f"iv_smile_bands_{pd.to_datetime(expiry).date()}")

def plot_mid_price_curves(df, expiry):
    dd = df[df["expiry"] == pd.to_datetime(expiry)].copy()
    if dd.empty: return
    fig = plt.figure()
    for side in ("C","P"):
        sub = dd[dd["side"]==side].sort_values("strike")
        if sub.empty: continue
        plt.plot(sub["strike"], sub["mid"], marker="o", linestyle="-", label=f"{side} mid")
        xx, yy = _smooth_xy(sub["strike"].values, sub["mid"].values, deg=3)
        plt.plot(xx, yy, linestyle="--")
    plt.axvline(float(S0), linestyle="--")
    plt.title(f"Option Mid Prices — {pd.to_datetime(expiry).date()}")
    plt.xlabel("Strike"); plt.ylabel("Price")
    plt.legend()
    _save_all(fig, OUT, f"mid_prices_{pd.to_datetime(expiry).date()}")

def plot_liquidity_heatmap(df):
    # grid of relative spread by (expiry, strike)
    dd = df.copy()
    dd["rel_spread"] = (dd["ask"] - dd["bid"]) / dd["mid"]
    dd = dd.replace([np.inf, -np.inf], np.nan).dropna(subset=["rel_spread","strike","expiry"])
    if dd.empty: return
    # pivot: rows=expiry, cols=strike
    piv = dd.pivot_table(index="expiry", columns="strike", values="rel_spread", aggfunc="median")
    piv = piv.sort_index().sort_index(axis=1)
    fig = plt.figure()
    plt.imshow(piv.values, aspect="auto", interpolation="nearest")
    plt.yticks(ticks=np.arange(len(piv.index)), labels=[pd.to_datetime(d).date() for d in piv.index])
    # fewer x ticks for readability
    xidx = np.linspace(0, piv.shape[1]-1, num=min(12, piv.shape[1]), dtype=int)
    plt.xticks(ticks=xidx, labels=[f"{piv.columns[i]:.0f}" for i in xidx], rotation=45)
    plt.title("Liquidity Heatmap — Relative Spread (median)")
    plt.xlabel("Strike"); plt.ylabel("Expiry")
    cb = plt.colorbar()
    cb.set_label("Rel. spread  (ask−bid)/mid")
    _save_all(fig, OUT, "liquidity_heatmap")

def plot_iv_term_structure(df, m_band=(0.95,1.05)):
    # average IV in a near-ATM band vs expiry + dispersion band
    dd = df[(df["moneyness"]>=m_band[0]) & (df["moneyness"]<=m_band[1])].copy()
    if dd.empty: return
    grp = dd.groupby("expiry")["iv_clamped"].agg(["mean","std","count"]).reset_index().sort_values("expiry")
    fig = plt.figure()
    plt.plot(grp["expiry"], grp["mean"], marker="o", linestyle="-", label="Mean IV (near-ATM)")
    # ±1 std shading in IV space (project as whiskers)
    for x, mu, sd in zip(grp["expiry"], grp["mean"], grp["std"]):
        if np.isfinite(sd):
            plt.vlines(x, mu-sd, mu+sd, linewidth=2)
    plt.xticks(rotation=45)
    plt.title(f"IV Term Structure (moneyness in [{m_band[0]}, {m_band[1]}])")
    plt.xlabel("Expiry"); plt.ylabel("Implied Volatility")
    plt.legend()
    _save_all(fig, OUT, f"iv_term_structure_{m_band[0]}_{m_band[1]}")

def plot_parity_residuals(df_pairs, expiry, tol=0.05):
    if df_pairs is None or len(df_pairs)==0: return
    dd = df_pairs[df_pairs["expiry"] == pd.to_datetime(expiry)].sort_values("strike")
    if dd.empty or ("parity_residual_rq" not in dd.columns and "parity_residual" not in dd.columns):
        return
    col = "parity_residual_rq" if "parity_residual_rq" in dd.columns else "parity_residual"
    fig = plt.figure()
    plt.plot(dd["strike"], dd[col], marker="o", linestyle="-", label="Residual")
    plt.axhline(0.0, linestyle="--")
    # tolerance band
    plt.fill_between(dd["strike"], -tol, tol, alpha=0.2)
    plt.title(f"Put–Call Parity Residuals — {pd.to_datetime(expiry).date()}")
    plt.xlabel("Strike"); plt.ylabel("Residual")
    plt.legend()
    _save_all(fig, OUT, f"parity_residuals_{pd.to_datetime(expiry).date()}")

def plot_iv_surface_tricontour(df):
    # T-moneyness IV surface (tricontour); sparse but effective for a prelim deck
    dd = df.dropna(subset=["T","moneyness","iv_clamped"]).copy()
    if dd.empty: return
    fig = plt.figure()
    tri = plt.tricontourf(dd["T"].values, dd["moneyness"].values, dd["iv_clamped"].values, levels=14)
    plt.xlabel("Maturity T (years)")
    plt.ylabel("Moneyness K/S0")
    plt.title("Implied Volatility Surface (tri-contour)")
    cb = plt.colorbar(tri)
    cb.set_label("IV")
    _save_all(fig, OUT, "iv_surface_tricontour")


In [ ]:

# ====== RUN ENGINE (caller) ======
# You can tweak these params as we add features (e.g., NSE fetcher, rate curves, etc.)
ticker = "AAPL"
dataset, S0 = build_option_dataset(
    ticker=ticker,
    expiries="auto",                # or a list like ["2025-12-19", "2026-03-20"]
    tmin=0.03, tmax=0.60,           # keep ~11 days to ~7 months
    moneyness_bounds=(0.7, 1.3),    # focus near-the-money
    iv_bounds=(0.01, 2.5),          # clamp IV into a sane band
    require_open_interest=True,
    save_csv=f"{ticker}_surface.csv"
)

print(f"S0 = {S0:.4f} | rows: {len(dataset)} | expiries: {dataset['expiry'].nunique()}")
print(dataset.groupby(["expiry", "side"])["strike"].count().unstack(fill_value=0).head())
display(dataset.head(10))

# ====== Run validation on the dataset you already built ======
# Assumptions for now; we can wire in curves later.
r_flat = 0.0   # risk-free rate (flat)
q_flat = 0.0   # dividend yield (flat)

checked, parity_pairs, summary = validate_dataset(dataset, S0=S0, r=r_flat, q=q_flat)

print("=== Data Quality Summary ===")
for k, v in summary.items():
    print(f"{k:>26}: {v}")

print("\nFirst 10 rows with BS reproduction error:")
display(checked[["expiry","side","strike","T","moneyness","iv_clamped","target_price","bs_repro","abs_err_vs_target"]].head(10))

print("\nPut-Call parity (first 10 pairs): residual should be near 0")
display(parity_pairs.head(10))

# ---- run inference + revalidation
rq_map = infer_r_q_per_expiry(dataset, S0)
print("Inferred r,q per expiry (first 5):")
for k, v in list(rq_map.items())[:5]:
    print(k.date(), v)

checked_rq, parity_pairs_rq, summary_rq = apply_r_q_and_check(dataset, S0, rq_map)

print("\n=== After r,q inference ===")
for k, v in summary_rq.items():
    print(f"{k:>28}: {v:.6f}" if isinstance(v, float) else f"{k:>28}: {v}")

print("\nParity residuals (first 10): should be near 0 now")
display(parity_pairs_rq[["expiry","T","strike","parity_residual_rq"]].head(10))

# Optional: filter out junk rows for calibration
clean_for_calib = checked_rq[
    (checked_rq["abs_err_vs_target_rq"] < 1.0) &                          # repro error below $1
    ((checked_rq["ask"] - checked_rq["bid"]) / checked_rq["mid"] < 0.3)   # spread < 30% of mid
].copy()

print("\nFinal calib rows:", len(clean_for_calib), "of", len(checked_rq))

# ---------- RUN ENGINE ----------
print("Saving preliminary, publication-grade diagnostics to", OUT)

# 1) Near-ATM term structure
plot_iv_term_structure(dataset, m_band=(0.95, 1.05))

# 2) Choose a handful of expiries (first 5 unique dates)
exp_list = sorted(dataset["expiry"].dropna().unique())[:5]
for ex in exp_list:
    # If you computed r,q per expiry earlier, parity_pairs_rq exists; otherwise, this will just skip.
    rq = None
    try:
        plot_iv_smile_with_bands(dataset, S0=S0, expiry=ex, r=0.0, q=0.0)  # you can inject inferred r,q later
    except Exception as e:
        print("iv_smile_with_bands skipped:", e)
    try:
        plot_mid_price_curves(dataset, ex)
    except Exception as e:
        print("mid_price_curves skipped:", e)
    try:
        plot_parity_residuals(parity_pairs_rq if 'parity_pairs_rq' in globals() else None, ex, tol=0.10)
    except Exception as e:
        print("parity_residuals skipped:", e)
    # liquidity scatter is nice, but heatmap is better globally

# 3) Global heatmap + global surface
plot_liquidity_heatmap(dataset)
plot_iv_surface_tricontour(dataset)

print("Done. Files saved in:", OUT.resolve())

S0 = 262.8200 | rows: 490 | expiries: 10
side         C   P
expiry            
2025-11-07  30  20
2025-11-14  29  21
2025-11-21  32  23
2025-11-28  29  18
2025-12-19  31  25


,underlying,expiry,T,side,type,strike,moneyness,bid,ask,mid,lastPrice,target_price,openInterest,volume,iv,iv_clamped,inTheMoney,currency
0,AAPL,2025-11-07,0.033364,C,1,185.0,0.703904,77.10,79.20,78.150,75.50,78.150,5,3.0,0.897462,0.897462,True,USD
1,AAPL,2025-11-07,0.033364,C,1,190.0,0.722928,72.20,74.15,73.175,68.45,73.175,14,4.0,0.847658,0.847658,True,USD
2,AAPL,2025-11-07,0.033364,C,1,195.0,0.741953,67.15,69.20,68.175,64.70,68.175,11,4.0,0.788576,0.788576,True,USD
3,AAPL,2025-11-07,0.033364,C,1,200.0,0.760977,62.70,64.05,63.375,62.70,63.375,727,738.0,0.791018,0.791018,True,USD
4,AAPL,2025-11-07,0.033364,C,1,205.0,0.780002,57.15,59.25,58.200,58.82,58.200,205,3.0,0.682132,0.682132,True,USD
5,AAPL,2025-11-07,0.033364,C,1,210.0,0.799026,52.30,54.20,53.250,53.80,53.250,164,148.0,0.639652,0.639652,True,USD
6,AAPL,2025-11-07,0.033364,C,1,215.0,0.818050,47.20,49.30,48.250,48.85,48.250,71,31.0,0.583012,0.583012,True,USD
7,AAPL,2025-11-07,0.033364,C,1,220.0,0.837075,42.35,44.30,43.325,44.60,43.325,380,30.0,0.543950,0.543950,True,USD
8,AAPL,2025-11-07,0.033364,C,1,225.0,0.856099,37.45,39.45,38.450,38.98,38.450,111,26.0,0.510747,0.510747,True,USD
9,AAPL,2025-11-07,0.033364,C,1,230.0,0.875124,33.20,34.55,33.875,33.86,33.875,330,26.0,0.512456,0.512456,True,USD


=== Data Quality Summary ===
                      rows: 490
    mean_abs_err_vs_target: 0.1563247920577353
  median_abs_err_vs_target: 0.06204401816072824
      pct_large_err_(>1.0): 1.2244897959183674
              parity_pairs: 212
      parity_residual_mean: 1.9314785248378543
    parity_residual_median: 1.5024926757812498

First 10 rows with BS reproduction error:


,expiry,side,strike,T,moneyness,iv_clamped,target_price,bs_repro,abs_err_vs_target
0,2025-11-07,C,185.0,0.033364,0.703904,0.897462,78.150,78.027584,0.122416
1,2025-11-07,C,190.0,0.033364,0.722928,0.847658,73.175,73.046089,0.128911
2,2025-11-07,C,195.0,0.033364,0.741953,0.788576,68.175,68.047270,0.127730
3,2025-11-07,C,200.0,0.033364,0.760977,0.791018,63.375,63.194575,0.180425
4,2025-11-07,C,205.0,0.033364,0.780002,0.682132,58.200,58.069103,0.130897
5,2025-11-07,C,210.0,0.033364,0.799026,0.639652,53.250,53.107438,0.142562
6,2025-11-07,C,215.0,0.033364,0.818050,0.583012,48.250,48.109960,0.140040
7,2025-11-07,C,220.0,0.033364,0.837075,0.543950,43.325,43.169491,0.155509
8,2025-11-07,C,225.0,0.033364,0.856099,0.510747,38.450,38.270819,0.179181
9,2025-11-07,C,230.0,0.033364,0.875124,0.512456,33.875,33.617804,0.257196



Put-Call parity (first 10 pairs): residual should be near 0


side,expiry,T,strike,C,P,parity_residual
0,2025-11-07,0.033364,185.0,78.150,0.025,0.304993
1,2025-11-07,0.033364,190.0,73.175,0.035,0.319993
2,2025-11-07,0.033364,195.0,68.175,0.050,0.304993
3,2025-11-07,0.033364,200.0,63.375,0.045,0.509993
4,2025-11-07,0.033364,205.0,58.200,0.065,0.314993
5,2025-11-07,0.033364,210.0,53.250,0.065,0.364993
6,2025-11-07,0.033364,215.0,48.250,0.090,0.339993
7,2025-11-07,0.033364,220.0,43.325,0.120,0.384993
8,2025-11-07,0.033364,225.0,38.450,0.160,0.469993
9,2025-11-07,0.033364,230.0,33.875,0.240,0.814993


Inferred r,q per expiry (first 5):
2025-11-07 {'T': 0.03336439401022258, 'DF': 0.9993067669172927, 'F': 263.25381956514207, 'r': 0.020784836663167622, 'q': -0.02864645509988897, 'n_pairs': 20}
2025-11-14 {'T': 0.05252934671863513, 'DF': 0.9992578947368417, 'F': 263.16429560884654, 'r': 0.014132685935582376, 'q': -0.010789005767796128, 'n_pairs': 20}
2025-11-21 {'T': 0.0716942974242021, 'DF': 1.0001166007905136, 'F': 263.32141113980776, 'r': -0.0016262659285145762, 'q': -0.02821089818791205, 'n_pairs': 23}
2025-11-28 {'T': 0.09085925038339417, 'DF': 0.9941568627450984, 'F': 263.69452881543134, 'r': 0.06449838787735433, 'q': 0.027937110013983378, 'n_pairs': 17}
2025-12-19 {'T': 0.14835411356478312, 'DF': 1.0015269230769233, 'F': 264.00038786929184, 'r': -0.010284571681451691, 'q': -0.040490389074158695, 'n_pairs': 25}

=== After r,q inference ===
                        rows: 490
   abs_err_vs_target_rq_mean: 0.865344
 abs_err_vs_target_rq_median: 0.401158
                parity_pairs: 2

side,expiry,T,strike,parity_residual_rq
0,2025-11-07,0.033364,185.0,-0.074571
1,2025-11-07,0.033364,190.0,-0.063038
2,2025-11-07,0.033364,195.0,-0.081504
3,2025-11-07,0.033364,200.0,0.120030
4,2025-11-07,0.033364,205.0,-0.078436
5,2025-11-07,0.033364,210.0,-0.031902
6,2025-11-07,0.033364,215.0,-0.060368
7,2025-11-07,0.033364,220.0,-0.018835
8,2025-11-07,0.033364,225.0,0.062699
9,2025-11-07,0.033364,230.0,0.404233



Final calib rows: 308 of 490
Saving preliminary, publication-grade diagnostics to figures_prelim
Done. Files saved in: /content/figures_prelim


In [ ]:
# ===============================================================
# CELL 8 — Compute Greeks and Visualize Risk Profiles (FIXED)
# ===============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
import os

# 1) Choose a source dataframe:
#    Preferred: checked_rq (has per-expiry r,q)
#    Fallback:  dataset     (we'll inject r,q = r_flat, q_flat)
if 'checked_rq' in globals():
    base_df = checked_rq.copy()
elif 'dataset' in globals():
    base_df = dataset.copy()
else:
    raise RuntimeError("No dataset found. Run Cell 7 first to create `dataset` or `checked_rq`.")

# 2) Ensure we have S (spot), r, q columns
if 'S0' not in globals():
    raise RuntimeError("S0 not in scope. Run Cell 7 first to set S0.")

base_df['S'] = float(S0)

# Use r,q from inference if available; otherwise fill with flat assumptions
if 'r' not in base_df.columns:
    r_flat = 0.0 if 'r_flat' not in globals() else float(r_flat)
    base_df['r'] = r_flat
if 'q' not in base_df.columns:
    q_flat = 0.0 if 'q_flat' not in globals() else float(q_flat)
    base_df['q'] = q_flat

# Sanity: make sure iv_clamped is numeric and positive where possible
base_df['iv_clamped'] = pd.to_numeric(base_df['iv_clamped'], errors='coerce')

# === Helper: Black–Scholes Greeks (annualized; Theta returned per year) ===
def bs_greeks(row):
    S   = float(row["S"])
    K   = float(row["strike"])
    T   = float(row["T"])
    r   = float(row["r"])
    q   = float(row["q"])
    vol = float(row["iv_clamped"]) if pd.notna(row["iv_clamped"]) else np.nan
    side = row["side"]

    # Edge cases
    if (T <= 0) or (not np.isfinite(vol)) or (vol <= 0) or (S <= 0) or (K <= 0):
        # Expired or invalid inputs → zero sensitivities
        return pd.Series([0.0, 0.0, 0.0, 0.0, 0.0], index=["delta","gamma","vega","theta","rho"])

    sqrtT = np.sqrt(T)
    d1 = (np.log(S/K) + (r - q + 0.5*vol*vol)*T) / (vol*sqrtT)
    d2 = d1 - vol*sqrtT
    Nd1 = norm.cdf(d1); Nd2 = norm.cdf(d2)
    n_d1 = norm.pdf(d1)

    # Delta
    if side == "C":
        delta = np.exp(-q*T) * Nd1
    else:  # Put
        delta = np.exp(-q*T) * (Nd1 - 1.0)

    # Gamma (same for calls/puts)
    gamma = np.exp(-q*T) * n_d1 / (S * vol * sqrtT)

    # Vega (per 1.0 of vol, not per vol point)
    vega = S * np.exp(-q*T) * n_d1 * sqrtT

    # Theta (per year). If you prefer per day, divide by 365 later where plotted.
    if side == "C":
        theta = (
            - (S * np.exp(-q*T) * n_d1 * vol) / (2.0 * sqrtT)
            - r * K * np.exp(-r*T) * Nd2
            + q * S * np.exp(-q*T) * Nd1
        )
    else:
        theta = (
            - (S * np.exp(-q*T) * n_d1 * vol) / (2.0 * sqrtT)
            + r * K * np.exp(-r*T) * norm.cdf(-d2)
            - q * S * np.exp(-q*T) * norm.cdf(-d1)
        )

    # Rho
    if side == "C":
        rho = K * T * np.exp(-r*T) * Nd2
    else:
        rho = -K * T * np.exp(-r*T) * norm.cdf(-d2)

    return pd.Series([delta, gamma, vega, theta, rho],
                     index=["delta","gamma","vega","theta","rho"])

# 3) Compute Greeks
df_greeks = base_df.copy()
df_greeks[["delta","gamma","vega","theta","rho"]] = df_greeks.apply(bs_greeks, axis=1)

# 4) Make output folder
os.makedirs("figures_risk", exist_ok=True)

# === Plot 1: Delta by Strike per expiry ===
for exp in sorted(df_greeks["expiry"].unique()):
    sub = df_greeks[df_greeks["expiry"] == exp].sort_values("strike")
    if sub.empty:
        continue
    plt.figure(figsize=(7,4))
    for side, marker in [("C","o"),("P","x")]:
        tmp = sub[sub["side"]==side]
        if not tmp.empty:
            plt.plot(tmp["strike"], tmp["delta"], marker=marker, linestyle="-", label=side)
    plt.axhline(0.0, linestyle="--", linewidth=0.8)
    plt.title(f"Delta vs Strike — {pd.to_datetime(exp).date()}")
    plt.xlabel("Strike"); plt.ylabel("Delta")
    plt.legend(); plt.grid(True); plt.tight_layout()
    plt.savefig(f"figures_risk/delta_{pd.to_datetime(exp).date()}.png", dpi=220)
    plt.close()

# === Plot 2: Gamma heatmap (expiry × strike) ===
pivot_gamma = df_greeks.pivot_table(index="expiry", columns="strike", values="gamma", aggfunc="mean")
if pivot_gamma.size > 0:
    plt.figure(figsize=(9,4))
    plt.imshow(pivot_gamma.values, aspect="auto", origin="lower")
    plt.colorbar(label="Gamma")
    # Tick labels
    ylabels = [pd.to_datetime(x).date() for x in pivot_gamma.index]
    plt.yticks(np.arange(len(ylabels)), ylabels)
    step = max(1, len(pivot_gamma.columns)//12)
    xlabels = [f"{x:.0f}" for x in pivot_gamma.columns[::step]]
    plt.xticks(np.arange(0, len(pivot_gamma.columns), step), xlabels, rotation=45)
    plt.title("Gamma Heatmap (Expiry × Strike)")
    plt.xlabel("Strike"); plt.ylabel("Expiry")
    plt.tight_layout()
    plt.savefig("figures_risk/gamma_heatmap.png", dpi=250)
    plt.close()

# === Plot 3: Vega vs Moneyness per expiry ===
for exp in sorted(df_greeks["expiry"].unique()):
    sub = df_greeks[df_greeks["expiry"] == exp].sort_values("moneyness")
    if sub.empty:
        continue
    plt.figure(figsize=(7,4))
    plt.plot(sub["moneyness"], sub["vega"], linestyle="-", marker=".")
    plt.title(f"Vega vs Moneyness — {pd.to_datetime(exp).date()}")
    plt.xlabel("Moneyness (K/S0)"); plt.ylabel("Vega")
    plt.grid(True); plt.tight_layout()
    plt.savefig(f"figures_risk/vega_{pd.to_datetime(exp).date()}.png", dpi=220)
    plt.close()

# === Optional: Theta ladder (plot per day for intuition) ===
theta_per_day = (df_greeks.groupby("expiry")["theta"].mean() / 365.0).sort_index()
if theta_per_day.size > 0:
    plt.figure(figsize=(8,4))
    theta_per_day.plot(kind="bar")
    plt.title("Average Theta per Expiry (per day)")
    plt.ylabel("Theta per day")
    plt.grid(True, axis="y"); plt.tight_layout()
    plt.savefig("figures_risk/theta_ladder.png", dpi=220)
    plt.close()

print("✅ Greeks computed and risk charts saved to ./figures_risk/")
df_greeks.head()

✅ Greeks computed and risk charts saved to ./figures_risk/


,underlying,expiry,T,side,type,strike,moneyness,bid,ask,mid,...,r,q,bs_repro_rq,abs_err_vs_target_rq,S,delta,gamma,vega,theta,rho
0,AAPL,2025-11-07,0.033364,C,1,185.0,0.703904,77.10,79.20,78.150,...,0.020785,-0.028646,78.401406,0.251406,262.820007,0.988199,0.000765,1.581294,-32.476106,6.049530
1,AAPL,2025-11-07,0.033364,C,1,190.0,0.722928,72.20,74.15,73.175,...,0.020785,-0.028646,73.422591,0.247591,262.820007,0.986444,0.000905,1.767306,-33.739480,6.200259
2,AAPL,2025-11-07,0.033364,C,1,195.0,0.741953,67.15,69.20,68.175,...,0.020785,-0.028646,68.426767,0.251767,262.820007,0.985384,0.001033,1.877736,-33.569799,6.357644
3,AAPL,2025-11-07,0.033364,C,1,200.0,0.760977,62.70,64.05,63.375,...,0.020785,-0.028646,63.573653,0.198653,262.820007,0.976751,0.001498,2.731100,-43.743187,6.443869
4,AAPL,2025-11-07,0.033364,C,1,205.0,0.780002,57.15,59.25,58.200,...,0.020785,-0.028646,58.453881,0.253881,262.820007,0.981694,0.001432,2.251672,-34.556372,6.658032


In [ ]:
# ===============================================================
# CELL 9 — IV Surface Fit + Scenario Pricing (FIXED & ENHANCED)
# ===============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from scipy.interpolate import UnivariateSpline, griddata
from scipy.special import erf  # <-- ADD THIS LINE
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (needed for 3D proj)

# Folders
os.makedirs("figures_surface", exist_ok=True)
os.makedirs("figures_scenarios", exist_ok=True)

# NEW (Correct)
def _norm_cdf(x):
    return 0.5 * (1.0 + erf(x / np.sqrt(2.0)))

def _bs_price(S, K, T, r, q, vol, side):
    """Black–Scholes price for a single option row."""
    if (T <= 0) or (vol <= 0) or (S <= 0) or (K <= 0):
        # intrinsic at expiry or invalid -> intrinsic value
        intrinsic = max(0.0, S*np.exp(-q*T) - K*np.exp(-r*T)) if side == "C" else max(0.0, K*np.exp(-r*T) - S*np.exp(-q*T))
        return intrinsic
    sqrtT = np.sqrt(T)
    d1 = (np.log(S/K) + (r - q + 0.5*vol*vol)*T) / (vol*sqrtT)
    d2 = d1 - vol*sqrtT
    if side == "C":
        return S*np.exp(-q*T)*_norm_cdf(d1) - K*np.exp(-r*T)*_norm_cdf(d2)
    else:
        return K*np.exp(-r*T)*_norm_cdf(-d2) - S*np.exp(-q*T)*_norm_cdf(-d1)

# ---- Source dataframe: use df_greeks (from Cell 8) -------------------------
if 'df_greeks' not in globals():
    raise RuntimeError("df_greeks not found. Run Cell 8 first.")

# Safety: ensure required columns exist
_need = {"expiry","T","strike","moneyness","iv_clamped","side","S","r","q","target_price"}
missing = _need - set(df_greeks.columns)
if missing:
    # We can still proceed if only target_price missing; otherwise data is incomplete
    print("Warning: missing columns in df_greeks:", missing)
# ===============================================================
# Step 1 — Fit a smooth IV curve per expiry (in MONEyness)
# ===============================================================
iv_fit_records = []
for exp, sub in df_greeks.groupby("expiry"):
    sub = sub.dropna(subset=["moneyness","iv_clamped"]).sort_values("moneyness")
    if len(sub) < 6:
        continue
    x = sub["moneyness"].values
    y = sub["iv_clamped"].astype(float).values

    # OLD, very small smoothing factor
    # s_val = 0.0001 * len(x)

    # <-- FIX: Use a much larger, fixed smoothing factor to handle noisy data
    s_val = 0.01

    try:
        # Remove duplicate x-values, which kill splines
        uniq_x, idx = np.unique(x, return_index=True)
        uniq_y = y[idx]

        if len(uniq_x) < 4: # Need at least 4 points for cubic
            continue

        spline = UnivariateSpline(uniq_x, uniq_y, s=s_val, k=3) # k=3 is cubic
    except Exception:
        try:
            # <-- FIX: Fallback to a *very* smooth line (linear)
            spline = UnivariateSpline(uniq_x, uniq_y, s=0.1, k=1)
        except Exception as e:
            print(f"Skipping expiry {exp} due to spline error: {e}")
            continue

    x_grid = np.linspace(x.min(), x.max(), 120)
    y_smooth = spline(x_grid)

    # Final check for nans
    if np.isnan(y_smooth).any():
        print(f"Skipping expiry {exp} due to NaNs in output spline.")
        continue

    iv_fit_records.append(pd.DataFrame({
        "expiry": exp,
        "T": float(sub["T"].iloc[0]),
        "moneyness": x_grid,
        "iv_fit": y_smooth
    }))
    ... # (rest of your plotting code in Cell 9 is fine)
    # plot raw vs fitted
    plt.figure(figsize=(7,4))
    plt.scatter(x, y, s=12, alpha=0.45, label="Raw IV")
    plt.plot(x_grid, y_smooth, label="Fitted IV", linewidth=2)
    plt.axvline(1.0, linestyle="--", linewidth=0.8)  # ATM line (K/S0 = 1)
    plt.title(f"IV Fit per Expiry — {pd.to_datetime(exp).date()}")
    plt.xlabel("Moneyness (K/S0)"); plt.ylabel("Implied Volatility")
    plt.legend(); plt.tight_layout()
    plt.savefig(f"figures_surface/iv_fit_{pd.to_datetime(exp).date()}.png", dpi=220)
    plt.close()

if not iv_fit_records:
    raise RuntimeError("No per-expiry IV fits produced (too few points per expiry?).")

iv_fit_df = pd.concat(iv_fit_records, ignore_index=True)

# ===============================================================
# Step 2 — Build a 3D IV surface: T × Moneyness → IV
# ===============================================================
# Grid in T (rows) and moneyness (cols)
T_vals = iv_fit_df["T"].values
M_vals = iv_fit_df["moneyness"].values
IV_vals = iv_fit_df["iv_fit"].values

T_grid, M_grid = np.mgrid[T_vals.min():T_vals.max():40j, M_vals.min():M_vals.max():80j]
IV_grid = griddata((T_vals, M_vals), IV_vals, (T_grid, M_grid), method="cubic")

# 3D surface
fig = plt.figure(figsize=(9,5))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(T_grid, M_grid, IV_grid, cmap="viridis", linewidth=0, antialiased=True)
ax.set_xlabel("Maturity T (years)")
ax.set_ylabel("Moneyness (K/S0)")
ax.set_zlabel("Implied Volatility")
ax.set_title("Smoothed IV Surface (T × Moneyness)")
plt.tight_layout()
plt.savefig("figures_surface/iv_surface_3D.png", dpi=250)
plt.close()

# 2D contour (nice for decks)
plt.figure(figsize=(7.5,4.5))
cs = plt.contourf(T_grid, M_grid, IV_grid, levels=14)
plt.xlabel("Maturity T (years)"); plt.ylabel("Moneyness (K/S0)")
plt.title("Smoothed IV Surface — Contour")
cbar = plt.colorbar(cs); cbar.set_label("IV")
plt.tight_layout()
plt.savefig("figures_surface/iv_surface_contour.png", dpi=250)
plt.close()

# ===============================================================
# Step 3 — Scenario Engine: spot / vol / time shocks → reprice BS
# ===============================================================
def scenario_reprice(df,
                     spot_shift_pct=0.00,         # e.g., +0.02 for +2%
                     vol_shift_abs=0.00,          # e.g., +0.01 for +1 vol point
                     days_forward=0):             # e.g., 1 day passes
    """
    Returns a new DataFrame with price_new and pnl per option, using BS with (r,q)
    and shocked (S, IV, T). Keeps other columns for plotting/analysis.
    """
    out = df.copy()
    # Shocked inputs
    S_new = out.get("S", pd.Series(S0, index=out.index)).astype(float) * (1.0 + float(spot_shift_pct))
    T_new = np.maximum(out["T"].astype(float) - float(days_forward)/365.25, 0.0)
    vol_new = (out["iv_clamped"].astype(float).fillna(0.0) + float(vol_shift_abs)).clip(lower=0.0001)

    # Price with BS per row
    # Price with BS per row
    out["price_new"] = [
        _bs_price(S_new.iat[i],
                  float(out["strike"].iat[i]),
                  float(T_new.iat[i]),  # <---- FIXED
                  float(out["r"].iat[i]) if "r" in out.columns else 0.0,
                  float(out["q"].iat[i]) if "q" in out.columns else 0.0,
                  float(vol_new.iat[i]), # <---- FIXED
                  out["side"].iat[i])
        for i in range(len(out))
    ]

    # Baseline price we compare against = your chosen anchor (“target_price” is best)
    anchor = "target_price" if "target_price" in out.columns else "mid"
    out["pnl"] = out["price_new"] - out[anchor].astype(float)
    out["S_shocked"] = S_new
    out["T_shocked"] = T_new
    out["iv_shocked"] = vol_new
    out["anchor_used"] = anchor
    return out

# Example scenario: +2% spot, +1 vol point (absolute), 0 days forward
df_scn = scenario_reprice(df_greeks, spot_shift_pct=0.02, vol_shift_abs=0.01, days_forward=0)

# PnL by strike (all expiries together, color by sign)
plt.figure(figsize=(8,4.5))
order = np.argsort(df_scn["strike"].values)
x = df_scn["strike"].values[order]
y = df_scn["pnl"].values[order]
colors = np.where(y >= 0, "tab:green", "tab:red")
plt.bar(x, y, color=colors)
plt.axhline(0.0, linestyle="--", linewidth=0.8)
plt.title("Scenario PnL per Contract — Spot +2%, Vol +1pt (abs)")
plt.xlabel("Strike"); plt.ylabel("PnL vs anchor")
plt.tight_layout()
plt.savefig("figures_scenarios/pnl_scenario_all.png", dpi=230)
plt.close()

# Optional: PnL bars per expiry (top few expiries) to see where risk concentrates
top_exps = sorted(df_scn["expiry"].unique())[:4]
for exp in top_exps:
    sub = df_scn[df_scn["expiry"] == exp].sort_values("strike")
    if sub.empty:
        continue
    plt.figure(figsize=(8,4))
    colors = np.where(sub["pnl"].values >= 0, "tab:green", "tab:red")
    plt.bar(sub["strike"].values, sub["pnl"].values, color=colors)
    plt.axhline(0.0, linestyle="--", linewidth=0.8)
    plt.title(f"Scenario PnL — {pd.to_datetime(exp).date()}  (Spot +2%, Vol +1pt)")
    plt.xlabel("Strike"); plt.ylabel("PnL vs anchor")
    plt.tight_layout()
    plt.savefig(f"figures_scenarios/pnl_{pd.to_datetime(exp).date()}.png", dpi=230)
    plt.close()

print("✅ IV surface & scenario charts saved to ./figures_surface and ./figures_scenarios/")

✅ IV surface & scenario charts saved to ./figures_surface and ./figures_scenarios/


In [ ]:
# ===============================================================
# CELL 10 — Trade Screener & Rankings
# ===============================================================
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

os.makedirs("figures_trade", exist_ok=True)

# Source = df_greeks (from Cell 8) enriched with spreads, efficiencies
screen = df_greeks.copy()

# Basic microstructure
screen["spread_abs"] = (screen["ask"] - screen["bid"]).clip(lower=0)
screen["spread_rel"] = screen["spread_abs"] / screen["mid"]
screen["liq_ok"]     = (screen["openInterest"].fillna(0) > 0) & (screen["spread_rel"] < 0.25)

# Efficiencies (add small epsilon to avoid divide-by-zero)
eps = 1e-8
price_base = (screen["mid"].abs() + eps)

screen["delta_eff"] = screen["delta"].abs() / price_base            # |Δ| per $
screen["vega_eff"]  = screen["vega"].abs()  / price_base            # Vega per $
screen["gamma_theta_ratio"] = np.where(screen["theta"]!=0, screen["gamma"]/(-screen["theta"]), np.nan)

# Composite score: weight what you care about (directional + vol + liquidity)
# You can tune weights later; keep it transparent
w_delta, w_vega, w_liq = 0.4, 0.4, 0.2
# Rank each metric within expiry to be fair across maturities
def zrank(s):
    r = s.rank(pct=True)
    return (r - r.mean()) / (r.std(ddof=0) + eps)

grp = screen.groupby("expiry")
screen["z_delta"] = grp["delta_eff"].transform(zrank)
screen["z_vega"]  = grp["vega_eff"].transform(zrank)
screen["z_liq"]   = grp["spread_rel"].transform(lambda s: -zrank(s))  # lower spread = better

screen["score"] = w_delta*screen["z_delta"] + w_vega*screen["z_vega"] + w_liq*screen["z_liq"]
screen["score"] = np.where(screen["liq_ok"], screen["score"], -999)   # hard-penalize illiquid

# Top picks per expiry
topN = 8
picks = (screen.sort_values(["expiry","score"], ascending=[True, False])
               .groupby("expiry").head(topN)
               .reset_index(drop=True))

# Save CSV + quick plot
picks.to_csv("trade_picks.csv", index=False)

for exp, sub in picks.groupby("expiry"):
    sub = sub.sort_values("score", ascending=True)
    plt.figure(figsize=(8,4))
    labels = [f'{s.side}{int(s.strike)}' for _, s in sub.iterrows()]
    plt.barh(labels, sub["score"].values)
    plt.title(f"Top {topN} Ranked Contracts — {pd.to_datetime(exp).date()}")
    plt.xlabel("Composite Score (↑ better)")
    plt.tight_layout()
    plt.savefig(f"figures_trade/picks_{pd.to_datetime(exp).date()}.png", dpi=240)
    plt.close()

print("✅ Trade screener done: trade_picks.csv + bar charts in ./figures_trade")

✅ Trade screener done: trade_picks.csv + bar charts in ./figures_trade


In [ ]:
# ===============================================================
# CELL 11 — Scenario Grid (Spot × Vol × Time) with Heatmaps
# ===============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

os.makedirs("figures_scenarios", exist_ok=True)

# Re-use scenario_reprice from Cell 9 and df_greeks as the base
spot_shocks = np.array([-0.05, -0.03, -0.01, 0.0, 0.01, 0.03, 0.05])   # -5% … +5%
vol_shocks  = np.array([-0.02, -0.01, 0.0, 0.01, 0.02])                 # -2pts … +2pts
days_fwd    = [0, 1, 5]   # same-day, +1d, +1w approx

# Choose a single expiry to visualize (start with nearest liquid)
focus_exp = sorted(df_greeks["expiry"].unique())[0]
base = df_greeks[df_greeks["expiry"]==focus_exp].copy()

def pnl_grid(one_row):
    """Return PnL grid for a single option across spot×vol at fixed time step."""
    grids = {}
    for d in days_fwd:
        mat = np.zeros((len(spot_shocks), len(vol_shocks)))
        for i, ds in enumerate(spot_shocks):
            for j, dv in enumerate(vol_shocks):
                tmp = scenario_reprice(one_row.to_frame().T, ds, dv, d)
                mat[i, j] = tmp["pnl"].iloc[0]
        grids[d] = mat
    return grids

# Build grids for a representative call/put around ATM
atm = base.iloc[(base["moneyness"]-1.0).abs().argsort()[:1]].copy()
if atm.empty:
    raise RuntimeError("No ATM row found for scenario grid.")
rowC = atm[atm["side"]=="C"].head(1)
rowP = atm[atm["side"]=="P"].head(1)
if rowC.empty: rowC = base[base["side"]=="C"].head(1)
if rowP.empty: rowP = base[base["side"]=="P"].head(1)

gridsC = pnl_grid(rowC.squeeze())
gridsP = pnl_grid(rowP.squeeze())

# Plot heatmaps
def heatmap(mat, title, fname):
    plt.figure(figsize=(6.5,4.8))
    im = plt.imshow(mat, aspect="auto", origin="lower")
    plt.colorbar(im, label="PnL per contract")
    plt.xticks(range(len(vol_shocks)), [f"{v:+.0%}pt" for v in vol_shocks])
    plt.yticks(range(len(spot_shocks)), [f"{s:+.0%}" for s in spot_shocks])
    plt.xlabel("Vol shift (absolute points)")
    plt.ylabel("Spot move")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(fname, dpi=230)
    plt.close()

for d in days_fwd:
    heatmap(gridsC[d], f"Call PnL Heatmap — {pd.to_datetime(focus_exp).date()} — +{d}d",
            f"figures_scenarios/heatmap_call_{pd.to_datetime(focus_exp).date()}_d{d}.png")
    heatmap(gridsP[d], f"Put PnL Heatmap — {pd.to_datetime(focus_exp).date()} — +{d}d",
            f"figures_scenarios/heatmap_put_{pd.to_datetime(focus_exp).date()}_d{d}.png")

print("✅ Scenario heatmaps saved in ./figures_scenarios")

✅ Scenario heatmaps saved in ./figures_scenarios


In [ ]:
# ===============================================================
# CELL 12 — Save a short run report (Markdown + CSV bundle)
# ===============================================================
from pathlib import Path
from datetime import datetime

report_dir = Path("run_report")
report_dir.mkdir(exist_ok=True)

# Key CSVs already exist; just copy references and write a markdown summary
summary_lines = []
summary_lines.append(f"# Option Surface Run Report ({datetime.utcnow().isoformat()}Z)\n")
summary_lines.append(f"- Underlying spot S0: **{S0:.2f}**")
summary_lines.append(f"- Rows in dataset: **{len(dataset)}**; expiries: **{dataset['expiry'].nunique()}**")
summary_lines.append(f"- Clean calib rows: **{len(df_greeks)}** (post-Greeks enrichment)")

summary_lines.append("\n## Files of interest")
summary_lines.append("- `trade_picks.csv` — ranked shortlist per expiry")
summary_lines.append("- `figures_prelim/*` — data sanity & diagnostics")
summary_lines.append("- `figures_risk/*` — Greeks & risk profiles")
summary_lines.append("- `figures_surface/*` — fitted IV surface")
summary_lines.append("- `figures_scenarios/*` — scenario PnL & heatmaps\n")

(summary_dir := report_dir / "figures").mkdir(exist_ok=True)
# (You can zip the folders if you like; leaving as pointers keeps it light.)

with open(report_dir / "README.md", "w") as f:
    f.write("\n".join(summary_lines))

print(f"✅ Report stub written to {report_dir.resolve()}")

✅ Report stub written to /content/run_report


/tmp/ipython-input-1975115516.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  summary_lines.append(f"# Option Surface Run Report ({datetime.utcnow().isoformat()}Z)\n")


In [ ]:
# ===============================================================
# CELL 13 — Heston Monte Carlo Engine (GPU-enabled - FINAL FIX)
# ===============================================================
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import time

# --- Setup: Use T4 GPU if available ---------------------------
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("⚠️ GPU not found, falling back to CPU.")

os.makedirs("figures_heston", exist_ok=True)

# --- Ensure df_greeks is loaded --------------------------------
if 'df_greeks' not in globals():
    raise RuntimeError("df_greeks not found. Run Cell 8 (or prior) first.")

# Get base parameters from our market data
try:
    S0 = df_greeks['S'].iloc[0]
    r = df_greeks['r'].iloc[0]
    q = df_greeks['q'].iloc[0]
except (KeyError, IndexError):
    print("Warning: Could not get S, r, q from df_greeks. Using dummy values.")
    S0, r, q = 100.0, 0.05, 0.0

# ===============================================================
# Step 1 — Define the Heston Path Simulator
# ===============================================================

def simulate_heston_paths(
    S0, v0, r, q, kappa, theta, sigma, rho,
    T, n_steps, n_paths,
    use_antithetic=True,
    seed=42,
    device=device
):
    """
    Simulates Heston paths for Spot (S) and Variance (v)
    using a stable log-Euler scheme for S and Full Truncation for v.

    Returns:
        S_paths (torch.Tensor): [n_paths, n_steps + 1]
        v_paths (torch.Tensor): [n_paths, n_steps + 1]
    """
    if seed is not None:
      torch.manual_seed(seed)
      if device.type == 'cuda':
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # for multi-GPU

    # --- FIX: Convert all scalar parameters to tensors on the device ---
    S0_t = torch.tensor(S0, device=device)
    v0_t = torch.tensor(v0, device=device)
    r_t = torch.tensor(r, device=device)
    q_t = torch.tensor(q, device=device)
    kappa_t = torch.tensor(kappa, device=device)
    theta_t = torch.tensor(theta, device=device)
    sigma_t = torch.tensor(sigma, device=device)
    rho_t = torch.tensor(rho, device=device)

    # Time step and derived constants
    dt = T / n_steps
    sqrt_dt = torch.tensor(dt, device=device).sqrt()
    # This was the specific source of the error
    sqrt_one_minus_rho_sq = (1.0 - rho_t**2).sqrt()
    # -----------------------------------------------------------------

    # Handle antithetic sampling
    if use_antithetic:
        if n_paths % 2 != 0:
            n_paths += 1  # Must be even
        n_gen = n_paths // 2
    else:
        n_gen = n_paths

    # Generate correlated random normals (Z1 for variance, Z2 for spot)
    Z1 = torch.randn(n_gen, n_steps, device=device)
    Z2 = torch.randn(n_gen, n_steps, device=device)

    if use_antithetic:
        # Concatenate Z with -Z to create antithetic pairs
        Z1 = torch.cat([Z1, -Z1], dim=0)
        Z2 = torch.cat([Z2, -Z2], dim=0)

    # dW_v = sqrt(dt) * Z1
    # dW_S = sqrt(dt) * (rho * Z1 + sqrt(1-rho^2) * Z2)
    dW_v = sqrt_dt * Z1
    # <-- FIX: Use the new tensors
    dW_S = sqrt_dt * (rho_t * Z1 + sqrt_one_minus_rho_sq * Z2)

    # Initialize path tensors
    X_paths = torch.zeros(n_paths, n_steps + 1, device=device)
    v_paths = torch.zeros(n_paths, n_steps + 1, device=device)

    # Set initial values
    X_paths[:, 0] = S0_t.log() # <-- FIX
    v_paths[:, 0] = v0_t       # <-- FIX

    # Use a small epsilon for numerical stability (Full Truncation)
    eps = torch.tensor(1e-10, device=device) # <-- FIX

    # Euler-Maruyama Loop
    for t in range(n_steps):
        # Get v_t, ensuring it's non-negative for the sqrt
        v_t = torch.maximum(v_paths[:, t], eps)
        sqrt_v_t = v_t.sqrt()

        # 1. Simulate Variance (Full Truncation Scheme)
        # <-- FIX: Use tensors for kappa, theta, sigma
        v_new = v_paths[:, t] + kappa_t * (theta_t - v_t) * dt + sigma_t * sqrt_v_t * dW_v[:, t]
        v_paths[:, t+1] = torch.maximum(v_new, eps) # Absorb at zero

        # 2. Simulate Log-Spot (Log-Euler Scheme)
        # dX = (r - q - 0.5*v_t)dt + sqrt(v_t)dW_S
        # <-- FIX: Use tensors for r, q
        X_paths[:, t+1] = X_paths[:, t] + (r_t - q_t - 0.5 * v_t) * dt + sqrt_v_t * dW_S[:, t]

    # Convert log-paths back to spot-paths
    S_paths = torch.exp(X_paths)

    return S_paths, v_paths

# ===============================================================
# Step 2 — Run Simulation & Plot Sample Paths
# ===============================================================

# Heston parameters (dummy values for now; Cell 16 will calibrate these)
v0    = 0.04    # Initial variance (e.g., 20% vol)
kappa = 2.0     # Speed of mean reversion
theta = 0.04    # Long-run variance
sigma = 0.3     # Vol of vol
rho   = -0.7    # Correlation (leverage effect)

# Simulation parameters
T       = 1.0       # 1 year maturity
n_steps = 252       # Daily steps
n_paths = 100_000   # 100k paths (fast on GPU)

print(f"Running Heston simulation...")
print(f"  Params: S0={S0:.2f}, v0={v0}, r={r}, q={q}, k={kappa}, th={theta}, sig={sigma}, rho={rho}")
print(f"  Config: T={T}, Steps={n_steps}, Paths={n_paths}, Antithetic=True")

start_time = time.time()
S_paths, v_paths = simulate_heston_paths(
    S0=S0, v0=v0, r=r, q=q,
    kappa=kappa, theta=theta, sigma=sigma, rho=rho,
    T=T, n_steps=n_steps, n_paths=n_paths,
    use_antithetic=True, device=device
)
end_time = time.time()

print(f"\nSimulation complete in {end_time - start_time:.4f} seconds.")
print(f"  S_paths shape: {S_paths.shape}")
print(f"  v_paths shape: {v_paths.shape}")

# --- Plot a few sample paths -----------------------------------
# Move data to CPU for plotting
n_plot = 10
S_cpu = S_paths.cpu().numpy()[:n_plot, :]
v_cpu = v_paths.cpu().numpy()[:n_plot, :]
time_axis = np.linspace(0, T, n_steps + 1)

plt.figure(figsize=(12, 5))

# Plot Spot Paths
plt.subplot(1, 2, 1)
plt.plot(time_axis, S_cpu.T, alpha=0.7)
plt.title(f"Heston Spot Paths (First {n_plot})")
plt.xlabel("Time T (years)")
plt.ylabel("Spot Price $S_t$")

# Plot Variance Paths
plt.subplot(1, 2, 2)
plt.plot(time_axis, v_cpu.T, alpha=0.7)
plt.axhline(theta, color='red', linestyle='--', linewidth=1.5, label=f"Long-run $\\theta={theta}$")
plt.title(f"Heston Variance Paths (First {n_plot})")
plt.xlabel("Time T (years)")
plt.ylabel("Variance $v_t$")
plt.legend()

plt.tight_layout()
plt.savefig("figures_heston/heston_sample_paths.png", dpi=250)
plt.close()

print(f"✅ Cell 13 complete. Sample paths saved to ./figures_heston/")

✅ Using GPU: Tesla T4
Running Heston simulation...
  Params: S0=262.82, v0=0.04, r=0.020784836663167622, q=-0.02864645509988897, k=2.0, th=0.04, sig=0.3, rho=-0.7
  Config: T=1.0, Steps=252, Paths=100000, Antithetic=True

Simulation complete in 0.1459 seconds.
  S_paths shape: torch.Size([100000, 253])
  v_paths shape: torch.Size([100000, 253])
✅ Cell 13 complete. Sample paths saved to ./figures_heston/


In [ ]:
# ===============================================================
# CELL 14 — Monte Carlo Option Pricer & BS Validation
# ===============================================================
import torch
import numpy as np
import pandas as pd
from scipy.special import erf # For BS validation
import time

# --- Ensure previous results are in memory ---------------------
if 'S_paths' not in globals() or 'v_paths' not in globals():
    raise RuntimeError("Heston paths (S_paths, v_paths) not found. Run Cell 13 first.")
if 'device' not in globals():
    raise RuntimeError("PyTorch 'device' not found. Run Cell 13 first.")

# --- Ensure base parameters are in memory ----------------------
try:
    S0 = df_greeks['S'].iloc[0]
    r = df_greeks['r'].iloc[0]
    q = df_greeks['q'].iloc[0]
except (KeyError, IndexError):
    print("Warning: Could not get S, r, q from df_greeks. Using dummy values.")
    S0, r, q = 100.0, 0.05, 0.0

# ===============================================================
# Step 1 — Define the Monte Carlo Pricer
# ===============================================================

def price_heston_mc(S_paths, K, T, r, side='C', device=device):
    """
    Prices a European option using Monte Carlo paths.

    Args:
        S_paths (torch.Tensor): [n_paths, n_steps + 1] asset paths
        K (float): Strike price
        T (float): Time to maturity (must match paths)
        r (float): Risk-free rate
        side (str): 'C' for Call, 'P' for Put

    Returns:
        torch.Tensor: A single (scalar) price
    """
    # Convert scalars to tensors on the GPU
    K_t = torch.tensor(K, device=device)
    T_t = torch.tensor(T, device=device)
    r_t = torch.tensor(r, device=device)

    # Get spot price at maturity (last column of S_paths)
    S_T = S_paths[:, -1]

    # Calculate payoffs
    if side == 'C':
        payoffs = torch.maximum(S_T - K_t, torch.tensor(0.0, device=device))
    elif side == 'P':
        payoffs = torch.maximum(K_t - S_T, torch.tensor(0.0, device=device))
    else:
        raise ValueError("side must be 'C' or 'P'")

    # Discount the average payoff
    price = torch.mean(payoffs) * torch.exp(-r_t * T_t)
    return price

# ===============================================================
# Step 2 — Price an Example Option using Heston Paths
# ===============================================================

# Use the Heston paths (S_paths) already in memory from Cell 13
# Let's price a 1-year, at-the-money (ATM) call
K_atm = S0
T_val = 1.0 # Must match the 'T' used to generate paths in Cell 13

print(f"Pricing ATM Call (K={K_atm:.2f}) using Heston paths from Cell 13...")

# Price the option
heston_price = price_heston_mc(S_paths, K=K_atm, T=T_val, r=r, side='C')

print(f"  > Heston MC Price: {heston_price.item():.4f}")


# ===============================================================
# Step 3 — Validation vs. Black-Scholes
# ===============================================================
# Heston collapses to Black-Scholes if:
# 1. sigma (vol-of-vol) = 0
# 2. v0 (initial variance) = theta (long-run variance)
# The MC price should match the BS price with vol = sqrt(v0)

print("\nValidating MC Pricer against Black-Scholes...")

# --- Analytical BS Pricer (from Cell 9, for comparison) ---
def _norm_cdf(x):
    return 0.5 * (1.0 + erf(x / np.sqrt(2.0)))

def _bs_price_cpu(S, K, T, r, q, vol, side):
    """Numpy/Scipy BS price for validation."""
    if (T <= 0) or (vol <= 0):
        intrinsic = max(0.0, S*np.exp(-q*T) - K*np.exp(-r*T)) if side == "C" else max(0.0, K*np.exp(-r*T) - S*np.exp(-q*T))
        return intrinsic
    sqrtT = np.sqrt(T)
    d1 = (np.log(S/K) + (r - q + 0.5*vol*vol)*T) / (vol*sqrtT)
    d2 = d1 - vol*sqrtT
    if side == "C":
        return S*np.exp(-q*T)*_norm_cdf(d1) - K*np.exp(-r*T)*_norm_cdf(d2)
    else:
        return K*np.exp(-r*T)*_norm_cdf(-d2) - S*np.exp(-q*T)*_norm_cdf(-d1)

# --- Run Validation Simulation ---
bs_vol = 0.20  # Test volatility
bs_v0 = bs_vol**2
bs_T = 1.0
bs_K = S0 # ATM strike

# BS Heston params: v0 = theta, sigma = 0
val_params = {
    'S0': S0, 'v0': bs_v0, 'r': r, 'q': q,
    'kappa': 2.0,     # Doesn't matter if v0=theta
    'theta': bs_v0,   # Must equal v0
    'sigma': 0.0,     # CRITICAL: No vol-of-vol
    'rho': 0.0,       # Doesn't matter if sigma=0
    'T': bs_T,
    'n_steps': 252,
    'n_paths': 200_000 # Use more paths for better accuracy
}

print(f"Simulating BS-equivalent paths (vol={bs_vol:.2f}, v0=theta={bs_v0:.2f}, sigma=0)...")
start_time = time.time()

S_paths_bs, v_paths_bs = simulate_heston_paths(
    **val_params, use_antithetic=True, device=device
)
mc_price_bs = price_heston_mc(S_paths_bs, K=bs_K, T=bs_T, r=r, side='C')

print(f"  > Simulation finished in {time.time() - start_time:.3f}s")

# Calculate analytical price
analytical_price_bs = _bs_price_cpu(S=S0, K=bs_K, T=bs_T, r=r, q=q, vol=bs_vol, side='C')

# --- Compare Results ---
price_diff = mc_price_bs.item() - analytical_price_bs
print(f"\n--- Validation Report ---")
print(f"  Analytical BS Price: {analytical_price_bs:.6f}")
print(f"  MC (Heston) Price:   {mc_price_bs.item():.6f}")
print(f"  Difference:          {price_diff:.6f}")

if abs(price_diff) < 0.01:
    print("✅ Validation PASSED. (Difference is negligible)")
else:
    print("⚠️ Validation FAILED. (Difference is significant)")

print("\n✅ Cell 14 complete.")

Pricing ATM Call (K=262.82) using Heston paths from Cell 13...
  > Heston MC Price: 28.0778

Validating MC Pricer against Black-Scholes...
Simulating BS-equivalent paths (vol=0.20, v0=theta=0.04, sigma=0)...
  > Simulation finished in 0.328s

--- Validation Report ---
  Analytical BS Price: 28.182594
  MC (Heston) Price:   28.188133
  Difference:          0.005539
✅ Validation PASSED. (Difference is negligible)

✅ Cell 14 complete.


In [ ]:
# ===============================================================
# CELL 14.5 — Clear Cached Validation Paths
# ===============================================================
# Run this cell to force Cell 15 to re-run the validation simulation
if 'S_paths_bs' in globals():
    del S_paths_bs
    print("Deleted 'S_paths_bs'. Cell 15 will now run the full simulation.")
else:
    print("Cache is clear. Ready to run Cell 15.")

Deleted 'S_paths_bs'. Cell 15 will now run the full simulation.


In [ ]:
# ===============================================================
# CELL 15 — Greek Estimation (Pathwise Δ, Finite-Difference V)
# ===============================================================
import torch
import numpy as np
import pandas as pd
from scipy.special import erf # For BS validation
import time

# --- FIX: Define `device` as a torch.device object ---
# This solves the "AttributeError: 'str' object has no attribute 'type'"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- Ensure previous results are in memory ---------------------
if 'S_paths' not in globals() or 'heston_price' not in globals():
    raise RuntimeError("Base paths (S_paths) and heston_price not found. Run Cells 13 & 14 first.")
if 'simulate_heston_paths' not in globals() or 'price_heston_mc' not in globals():
    raise RuntimeError("Helper functions (simulate_heston_paths, price_heston_mc) not found. Run Cells 13 & 14 first.")

# --- Analytical BS Helper Functions (CPU) ---
# These are included here to make the cell self-contained and prevent errors.
def _norm_cdf(x):
    """Standard Normal CDF (uses erf imported from scipy)"""
    return (1.0 + erf(x / np.sqrt(2.0))) / 2.0

def _bs_delta_cpu(S, K, T, r, q, vol, side):
    if (T <= 0) or (vol <= 0):
        return 1.0 if (S > K and side == 'C') else (-1.0 if S < K and side == 'P' else 0.0)
    sqrtT = np.sqrt(T)
    d1 = (np.log(S/K) + (r - q + 0.5*vol*vol)*T) / (vol*sqrtT)
    if side == 'C':
        return np.exp(-q*T) * _norm_cdf(d1)
    else:
        return -np.exp(-q*T) * _norm_cdf(-d1)

def _bs_vega_cpu(S, K, T, r, q, vol, side):
    if (T <= 0) or (vol <= 0): return 0.0
    sqrtT = np.sqrt(T)
    d1 = (np.log(S/K) + (r - q + 0.5*vol*vol)*T) / (vol*sqrtT)
    d1_pdf = (1.0 / np.sqrt(2.0 * np.pi)) * np.exp(-0.5 * d1 * d1)
    return S * np.exp(-q*T) * d1_pdf * sqrtT
# -----------------------------------------------------------------

# --- Ensure base parameters are in memory ----------------------
try:
    # Assuming df_greeks is loaded in a previous cell
    S0 = df_greeks['S'].iloc[0]
    r = df_greeks['r'].iloc[0]
    q = df_greeks['q'].iloc[0]
except Exception:
    print("Warning: Could not get S, r, q from df_greeks. Using dummy values.")
    S0, r, q = 100.0, 0.05, 0.0

# --- Heston parameters from Cell 13 (for FD Vega) --------------
v0    = 0.04    # Initial variance
kappa = 2.0     # Speed of mean reversion
theta = 0.04    # Long-run variance
sigma = 0.3     # Vol of vol
rho   = -0.7    # Correlation
T     = 1.0     # 1 year maturity
n_steps = 252   # Daily steps
n_paths = 100_000 # For Heston Greeks, not validation

# ===============================================================
# Step 1 — Define Pathwise Delta Function
# ===============================================================
def calculate_pathwise_delta(S_paths, S0, K, T, r, side='C', device=device):
    K_t, T_t, r_t, S0_t = torch.tensor(K, device=device), torch.tensor(T, device=device), torch.tensor(r, device=device), torch.tensor(S0, device=device)
    S_T = S_paths[:, -1]
    payoff_deriv = (S_T > K_t).float() if side == 'C' else (S_T < K_t).float() * -1.0
    path_deriv = S_T / S0_t
    return torch.mean(payoff_deriv * path_deriv) * torch.exp(-r_t * T_t)

# ===============================================================
# Step 2 — Define Finite-Difference Vega Function
# ===============================================================
def calculate_fd_vega(base_price, S0, v0, r, q, kappa, theta, sigma, rho,
                      T, n_steps, n_paths, K, side='C',
                      v0_bump=0.001, seed=42, device=device):
    print(f"  > Bumping v0 from {v0:.4f} to {v0 + v0_bump:.4f} for FD Vega...")
    S_paths_bump, _ = simulate_heston_paths(
        S0=S0, v0=(v0 + v0_bump), r=r, q=q, kappa=kappa, theta=theta, sigma=sigma, rho=rho,
        T=T, n_steps=n_steps, n_paths=n_paths, use_antithetic=True, seed=seed, device=device
    )
    price_bump = price_heston_mc(S_paths_bump, K=K, T=T, r=r, side=side)
    del S_paths_bump
    torch.cuda.empty_cache()
    vega_dv0 = (price_bump - base_price) / v0_bump
    return vega_dv0 * (2.0 * np.sqrt(v0))

# ===============================================================
# Step 3 — Calculate Greeks for Heston Paths
# ===============================================================
heston_price_float = heston_price.item() if torch.is_tensor(heston_price) else heston_price
K_atm = S0
print(f"Calculating Greeks for ATM Call (K={K_atm:.2f}) using Heston paths...")

start_time = time.time()
heston_delta = calculate_pathwise_delta(S_paths, S0, K_atm, T, r, side='C')
print(f"  > Pathwise Delta:    {heston_delta.item():.6f} (calc in {time.time()-start_time:.4f}s)")

start_time = time.time()
heston_vega = calculate_fd_vega(
    heston_price_float, S0, v0, r, q, kappa, theta, sigma, rho,
    T, n_steps, n_paths, K_atm, side='C', seed=42
)
print(f"  > Finite-Diff Vega:  {heston_vega.item():.6f} (calc in {time.time()-start_time:.4f}s)")

# ===============================================================
# Step 4 — Validation vs. Black-Scholes Greeks
# ===============================================================
print(f"\n--- Validating Greeks against Black-Scholes ---")

# ---
# FIX: Do NOT clear global S_paths/v_paths here.
# This allows subsequent cells to use the base Heston run from Cell 13/14.
# VRAM for *this cell's validation* will be cleared locally.
# ---
print("  > Proceeding with validation...")

# --- Parameters for Validation ---
bs_vol = 0.20
bs_v0 = bs_vol**2
bs_T = 1.0
bs_K = S0

# --- Compare Delta (Fast) ---
n_paths_delta = 200_000
delta_val_params = { 'S0': S0, 'v0': bs_v0, 'r': r, 'q': q, 'kappa': 2.0, 'theta': bs_v0, 'sigma': 0.0, 'rho': 0.0, 'T': bs_T, 'n_steps': 252, 'n_paths': n_paths_delta }
S_paths_bs_delta, _ = simulate_heston_paths(**delta_val_params, use_antithetic=True, device=device)
mc_delta_bs = calculate_pathwise_delta(S_paths_bs_delta, S0, bs_K, bs_T, r, side='C')
analytical_delta_bs = _bs_delta_cpu(S0, bs_K, bs_T, r, q, bs_vol, side='C')
print(f"\n[Delta Validation]")
print(f"  Analytical BS Delta: {analytical_delta_bs:.6f}")
print(f"  MC Pathwise Delta:   {mc_delta_bs.item():.6f} (using {n_paths_delta:,} paths)")

mc_delta_value = mc_delta_bs.item()
analytical_delta_value = analytical_delta_bs
print(f"  Difference:          {mc_delta_value - analytical_delta_value:.6f}")

print("\n  > Clearing Delta validation tensors from VRAM...")
del S_paths_bs_delta, mc_delta_bs
torch.cuda.empty_cache()
print("  > VRAM cleared. Starting Vega batch loop.")

# --- Compare Vega (Slow, using BATCHING) ---
print(f"\n[Vega Validation] (Using high-precision batching...)")

n_total_paths = 200_000_000
n_paths_per_loop = 1_500_000
# Use ceiling division to ensure all paths are covered
n_loops = (n_total_paths + n_paths_per_loop - 1) // n_paths_per_loop
print(f"  Total paths: {n_total_paths:,}")
print(f"  Running {n_loops} loops of {n_paths_per_loop:,} paths each...")

base_prices, bump_prices = [], []
v0_bump = 0.001
base_params = { 'S0': S0, 'v0': bs_v0, 'r': r, 'q': q, 'kappa': 2.0, 'theta': bs_v0, 'sigma': 0.0, 'rho': 0.0, 'T': bs_T, 'n_steps': n_steps, 'n_paths': n_paths_per_loop }
bump_params = {**base_params, 'v0': bs_v0 + v0_bump, 'theta': bs_v0 + v0_bump}

start_vega_loop = time.time()
for i in range(n_loops):
    loop_seed = 42 + i

    # --- FIX: Process one simulation at a time to conserve VRAM ---
    # 1. Price Base
    S_base, _ = simulate_heston_paths(**base_params, seed=loop_seed, device=device, use_antithetic=True)
    P_base = price_heston_mc(S_base, K=bs_K, T=bs_T, r=r, side='C')
    base_prices.append(P_base.item())
    del S_base, P_base
    torch.cuda.empty_cache() # Clean up immediately

    # 2. Price Bump
    S_bump, _ = simulate_heston_paths(**bump_params, seed=loop_seed, device=device, use_antithetic=True)
    P_bump = price_heston_mc(S_bump, K=bs_K, T=bs_T, r=r, side='C')
    bump_prices.append(P_bump.item())
    del S_bump, P_bump
    torch.cuda.empty_cache() # Clean up immediately

    if (i+1) % 10 == 0 or (i+1) == n_loops:
        print(f"    ...completed loop {i+1}/{n_loops}")

print(f"  > Batch simulation finished in {time.time() - start_vega_loop:.2f}s")

# Calculate final results
avg_price_base = np.mean(base_prices)
avg_price_bump = np.mean(bump_prices)
vega_dv0_bs = (avg_price_bump - avg_price_base) / v0_bump
mc_vega_bs = vega_dv0_bs * (2.0 * np.sqrt(bs_v0))
analytical_vega_bs = _bs_vega_cpu(S0, bs_K, bs_T, r, q, bs_vol, side='C')

print(f"\n[Vega Validation Results]")
print(f"  Analytical BS Vega: {analytical_vega_bs:.6f}")
print(f"  MC FD Vega:         {mc_vega_bs:.6f} (using {n_total_paths:,} total paths)")
print(f"  Difference:         {mc_vega_bs - analytical_vega_bs:.6f}")

# Final check using the stored scalar values
if abs(mc_delta_value - analytical_delta_value) < 0.001 and abs(mc_vega_bs - analytical_vega_bs) < 0.1:
    print("\n✅ Greek Validation PASSED.")
else:
    print("\n⚠️ Greek Validation FAILED.")

print("\n✅ Cell 15 (FD) complete.")

Using device: cuda
Calculating Greeks for ATM Call (K=262.82) using Heston paths...
  > Pathwise Delta:    0.711983 (calc in 0.0485s)
  > Bumping v0 from 0.0400 to 0.0410 for FD Vega...
  > Finite-Diff Vega:  42.185000 (calc in 0.1753s)

--- Validating Greeks against Black-Scholes ---

  > Clearing VRAM before validation...
  > VRAM cleared.

[Delta Validation]
  Analytical BS Delta: 0.654239
  MC Pathwise Delta:   0.654396 (using 200,000 paths)
  Difference:          0.000157

  > Clearing Delta validation tensors from VRAM...
  > VRAM cleared. Starting Vega batch loop.

[Vega Validation] (Using high-precision batching...)
  Total paths: 200,000,000
  Running 134 loops of 1,500,000 paths each...
    ...completed loop 10/134
    ...completed loop 20/134
    ...completed loop 30/134
    ...completed loop 40/134
    ...completed loop 50/134
    ...completed loop 60/134
    ...completed loop 70/134
    ...completed loop 80/134
    ...completed loop 90/134
    ...completed loop 100/134
   

In [ ]:
# ===============================================================
# CELL 15 — Greek Estimation (Pathwise Δ, Finite-Difference V)
# ===============================================================
import torch
import numpy as np
import pandas as pd
from scipy.special import erf # For BS validation
import time

# --- FIX: Define `device` as a torch.device object ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- Ensure previous results are in memory ---------------------
if 'S_paths' not in globals() or 'heston_price' not in globals():
    raise RuntimeError("Base paths (S_paths) and heston_price not found. Run Cells 13 & 14 first.")
if 'simulate_heston_paths' not in globals() or 'price_heston_mc' not in globals():
    raise RuntimeError("Helper functions (simulate_heston_paths, price_heston_mc) not found. Run Cells 13 & 14 first.")

# --- Analytical BS Helper Functions (CPU) ---
def _norm_cdf(x):
    return (1.0 + erf(x / np.sqrt(2.0))) / 2.0

def _bs_delta_cpu(S, K, T, r, q, vol, side):
    if (T <= 0) or (vol <= 0): return 1.0 if (S > K and side == 'C') else (-1.0 if S < K and side == 'P' else 0.0)
    sqrtT = np.sqrt(T); d1 = (np.log(S/K) + (r - q + 0.5*vol*vol)*T) / (vol*sqrtT)
    if side == 'C': return np.exp(-q*T) * _norm_cdf(d1)
    else: return -np.exp(-q*T) * _norm_cdf(-d1)

def _bs_vega_cpu(S, K, T, r, q, vol, side):
    if (T <= 0) or (vol <= 0): return 0.0
    sqrtT = np.sqrt(T); d1 = (np.log(S/K) + (r - q + 0.5*vol*vol)*T) / (vol*sqrtT)
    d1_pdf = (1.0 / np.sqrt(2.0 * np.pi)) * np.exp(-0.5 * d1 * d1)
    return S * np.exp(-q*T) * d1_pdf * sqrtT
# -----------------------------------------------------------------

# --- Ensure base parameters are in memory ----------------------
try:
    S0 = df_greeks['S'].iloc[0]; r = df_greeks['r'].iloc[0]; q = df_greeks['q'].iloc[0]
except Exception:
    print("Warning: Could not get S, r, q from df_greeks. Using dummy values.")
    S0, r, q = 100.0, 0.05, 0.0

# --- Heston parameters from Cell 13 (for FD Vega) --------------
v0 = 0.04; kappa = 2.0; theta = 0.04; sigma = 0.3; rho = -0.7
T = 1.0; n_steps = 252; n_paths = 100_000

# ===============================================================
# Step 1 — Define Pathwise Delta & CORRECTED Pathwise Vega
# ===============================================================
def calculate_pathwise_delta(S_paths, S0, K, T, r, side='C', device=device):
    K_t, T_t, r_t, S0_t = torch.tensor(K, device=device), torch.tensor(T, device=device), torch.tensor(r, device=device), torch.tensor(S0, device=device)
    S_T = S_paths[:, -1]
    payoff_deriv = (S_T > K_t).float() if side == 'C' else (S_T < K_t).float() * -1.0
    path_deriv = S_T / S0_t
    return torch.mean(payoff_deriv * path_deriv) * torch.exp(-r_t * T_t)

# FIX A: Corrected Pathwise Vega Function
def calculate_pathwise_vega(S_paths, S0, K, T, r, q, vol, side='C', device=device):
    """
    Calculates Pathwise Vega for a Black-Scholes process.
    FIX: This formula *must* account for the dividend yield 'q'.
    """
    K_t = torch.tensor(K, device=device)
    T_t = torch.tensor(T, device=device)
    r_t = torch.tensor(r, device=device)
    q_t = torch.tensor(q, device=device) # <-- FIX A.1: Add q_t
    S0_t = torch.tensor(S0, device=device)
    vol_t = torch.tensor(vol, device=device)

    S_T = S_paths[:, -1]
    log_ST_S0 = torch.log(S_T / S0_t)

    # FIX A.2: Correctly back out W_T using the (r - q) drift
    W_T = (log_ST_S0 - (r_t - q_t - 0.5 * vol_t**2) * T_t) / vol_t

    # Path derivative (dS_T / d(vol)) is S_T * (-vol * T + W_T)
    path_deriv = S_T * (-vol_t * T_t + W_T)

    # Payoff derivative d(Payoff) / d(S_T)
    payoff_deriv = (S_T > K_t).float() if side == 'C' else (S_T < K_t).float() * -1.0

    # Chain rule: [d(Payoff)/d(S_T)] * [d(S_T)/d(vol)]
    vega = torch.mean(payoff_deriv * path_deriv) * torch.exp(-r_t * T_t)
    return vega

# ===============================================================
# Step 2 — Define Finite-Difference Vega Function (for HESTON)
# ===============================================================
def calculate_fd_vega(base_price, S0, v0, r, q, kappa, theta, sigma, rho,
                      T, n_steps, n_paths, K, side='C',
                      v0_bump=0.001, seed=42, device=device):
    print(f"  > Bumping v0 from {v0:.4f} to {v0 + v0_bump:.4f} for FD Vega...")
    S_paths_bump, _ = simulate_heston_paths(
        S0=S0, v0=(v0 + v0_bump), r=r, q=q, kappa=kappa, theta=theta, sigma=sigma, rho=rho,
        T=T, n_steps=n_steps, n_paths=n_paths, use_antithetic=True, seed=seed, device=device
    )
    price_bump = price_heston_mc(S_paths_bump, K=K, T=T, r=r, side=side)
    del S_paths_bump; torch.cuda.empty_cache()
    vega_dv0 = (price_bump - base_price) / v0_bump
    return vega_dv0 * (2.0 * np.sqrt(v0))

# ===============================================================
# Step 3 — Calculate Greeks for Heston Paths
# ===============================================================
heston_price_float = heston_price.item() if torch.is_tensor(heston_price) else heston_price
K_atm = S0
print(f"Calculating Greeks for ATM Call (K={K_atm:.2f}) using Heston paths...")

start_time = time.time()
heston_delta = calculate_pathwise_delta(S_paths, S0, K_atm, T, r, side='C')
print(f"  > Pathwise Delta:    {heston_delta.item():.6f} (calc in {time.time()-start_time:.4f}s)")

start_time = time.time()
heston_vega = calculate_fd_vega(
    heston_price_float, S0, v0, r, q, kappa, theta, sigma, rho,
    T, n_steps, n_paths, K_atm, side='C', seed=42
)
print(f"  > Finite-Diff Vega:  {heston_vega.item():.6f} (calc in {time.time()-start_time:.4f}s)")

# ===============================================================
# Step 4 — Validation vs. Black-Scholes Greeks (Corrected)
# ===============================================================
print(f"\n--- Validating Greeks against Black-Scholes ---")
print("\n  > Clearing VRAM before validation...")
if 'S_paths' in globals(): del S_paths
if 'v_paths' in globals(): del v_paths
torch.cuda.empty_cache(); print("  > VRAM cleared.")

# --- Parameters for Validation ---
bs_vol = 0.20; bs_v0 = bs_vol**2; bs_T = 1.0; bs_K = S0
bs_params = { 'S0': S0, 'v0': bs_v0, 'r': r, 'q': q, 'kappa': 2.0, 'theta': bs_v0, 'sigma': 0.0, 'rho': 0.0, 'T': bs_T, 'n_steps': 252, 'use_antithetic': True, 'device': device}

# --- Shared Parameters for Batching ---
total_paths_target = 200_000_000
batch_size = 1_500_000
n_loops = (total_paths_target + batch_size - 1) // batch_size # Ceiling division

# FIX B.1: Calculate the *actual* number of paths that will be simulated
actual_total_paths = n_loops * batch_size

# --- Compare Delta (Batched, Pathwise) ---
print(f"\n[Delta Validation] (Using batched pathwise method...)")
print(f"  Target paths: {total_paths_target:,}, Batch size: {batch_size:,}, Loops: {n_loops}")
print(f"  Actual paths: {actual_total_paths:,}")
total_delta_sum = 0.0
start_delta_loop = time.time()

for i in range(n_loops):
    S_paths_batch, _ = simulate_heston_paths(**bs_params, n_paths=batch_size, seed=(42 + i))
    mc_delta_batch = calculate_pathwise_delta(S_paths_batch, S0, bs_K, bs_T, r, side='C')
    total_delta_sum += mc_delta_batch.item() * batch_size
    del S_paths_batch, mc_delta_batch
    torch.cuda.empty_cache()

# FIX B.2: Divide by the *actual* total paths
mc_delta_bs = total_delta_sum / actual_total_paths
analytical_delta_bs = _bs_delta_cpu(S0, bs_K, bs_T, r, q, bs_vol, side='C')
print(f"  Loop finished in {time.time() - start_delta_loop:.2f}s")
print(f"  Analytical BS Delta: {analytical_delta_bs:.6f}")
print(f"  MC Pathwise Delta:   {mc_delta_bs:.6f} (using {actual_total_paths:,} paths)")
mc_delta_value = mc_delta_bs; analytical_delta_value = analytical_delta_bs
print(f"  Difference:          {mc_delta_value - analytical_delta_value:.6f}")


# --- Compare Vega (Batched, Pathwise) ---
print(f"\n[Vega Validation] (Using batched pathwise method...)")
print(f"  Actual paths: {actual_total_paths:,}, Batch size: {batch_size:,}, Loops: {n_loops}")
total_vega_sum = 0.0
start_vega_loop = time.time()
for i in range(n_loops):
    S_paths_batch, _ = simulate_heston_paths(**bs_params, n_paths=batch_size, seed=(99 + i)) # Use different seed

    # FIX A.3: Call the *corrected* function with 'q'
    mc_vega_batch = calculate_pathwise_vega(S_paths_batch, S0, bs_K, bs_T, r, q, bs_vol, side='C')

    total_vega_sum += mc_vega_batch.item() * batch_size
    del S_paths_batch, mc_vega_batch
    torch.cuda.empty_cache()

# FIX B.2: Divide by the *actual* total paths
mc_vega_bs = total_vega_sum / actual_total_paths
analytical_vega_bs = _bs_vega_cpu(S0, bs_K, bs_T, r, q, bs_vol, side='C')
print(f"  Loop finished in {time.time() - start_vega_loop:.2f}s")
print(f"  Analytical BS Vega: {analytical_vega_bs:.6f}")
print(f"  MC Pathwise Vega:   {mc_vega_bs:.6f} (using {actual_total_paths:,} paths)")
print(f"  Difference:         {mc_vega_bs - analytical_vega_bs:.6f}")

# --- Final Combined Validation ---
if abs(mc_delta_value - analytical_delta_value) < 0.001 and abs(mc_vega_bs - analytical_vega_bs) < 0.1:
    print("\n✅ Greek Validation PASSED.")
else:
    print("\n⚠️ Greek Validation FAILED.")

print("\n✅ Cell 15 complete.")

Using device: cuda
Calculating Greeks for ATM Call (K=262.82) using Heston paths...
  > Pathwise Delta:    0.711983 (calc in 0.0013s)
  > Bumping v0 from 0.0400 to 0.0410 for FD Vega...
  > Finite-Diff Vega:  42.185000 (calc in 0.1943s)

--- Validating Greeks against Black-Scholes ---

  > Clearing VRAM before validation...
  > VRAM cleared.

[Delta Validation] (Using batched pathwise method...)
  Target paths: 200,000,000, Batch size: 1,500,000, Loops: 134
  Actual paths: 201,000,000
  Loop finished in 335.82s
  Analytical BS Delta: 0.654239
  MC Pathwise Delta:   0.654241 (using 201,000,000 paths)
  Difference:          0.000003

[Vega Validation] (Using batched pathwise method...)
  Actual paths: 201,000,000, Batch size: 1,500,000, Loops: 134
  Loop finished in 336.52s
  Analytical BS Vega: 101.587301
  MC Pathwise Vega:   101.600547 (using 201,000,000 paths)
  Difference:         0.013246

✅ Greek Validation PASSED.

✅ Cell 15 complete.


In [ ]:
# ===============================================================
# CELL 16 — Heston Model Calibration (STABLE FIX)
# ===============================================================
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import time
from scipy.optimize import minimize, brentq
from scipy.special import erf

# --- Ensure all previous helpers are loaded --------------------
if 'simulate_heston_paths' not in globals() or 'price_heston_mc' not in globals():
    raise RuntimeError("MC helpers (simulate_heston_paths, price_heston_mc) not found. Run Cells 13 & 14.")
if 'iv_fit_df' not in globals():
    raise RuntimeError("Smoothed IV surface (iv_fit_df) not found. Run Cell 9.")
if 'device' not in globals():
    raise RuntimeError("PyTorch 'device' not found. Run Cell 13 first.")

# --- Ensure base parameters are in memory ----------------------
try:
    S0 = df_greeks['S'].iloc[0]
    r = df_greeks['r'].iloc[0]
    q = df_greeks['q'].iloc[0]
except (KeyError, IndexError):
    print("Warning: Could not get S, r, q from df_greeks. Using dummy values.")
    S0, r, q = 100.0, 0.05, 0.0

os.makedirs("figures_calibration", exist_ok=True)

# ===============================================================
# Step 1 — Helpers for IV Calculation
# ===============================================================

# --- Analytical BS Pricer (from Cell 14) ---
def _norm_cdf(x):
    return 0.5 * (1.0 + erf(x / np.sqrt(2.0)))

def _bs_price_cpu(S, K, T, r, q, vol, side):
    """Numpy/Scipy BS price for validation."""
    if (T <= 0):
        intrinsic = max(0.0, S*np.exp(-q*T) - K*np.exp(-r*T)) if side == "C" else max(0.0, K*np.exp(-r*T) - S*np.exp(-q*T))
        return intrinsic
    if (vol <= 1e-6): # Handle zero vol
        return max(0.0, S*np.exp(-q*T) - K*np.exp(-r*T)) if side == "C" else max(0.0, K*np.exp(-r*T) - S*np.exp(-q*T))

    sqrtT = np.sqrt(T)
    d1 = (np.log(S/K) + (r - q + 0.5*vol*vol)*T) / (vol*sqrtT)
    d2 = d1 - vol*sqrtT
    if side == "C":
        return S*np.exp(-q*T)*_norm_cdf(d1) - K*np.exp(-r*T)*_norm_cdf(d2)
    else:
        return K*np.exp(-r*T)*_norm_cdf(-d2) - S*np.exp(-q*T)*_norm_cdf(-d1)

def calculate_bs_iv(target_price, S, K, T, r, q, side):
    """Finds the IV that matches a target price using a root finder."""
    def price_error(vol):
        return _bs_price_cpu(S, K, T, r, q, vol, side) - target_price

    intrinsic = max(0.0, (S*np.exp(-q*T) - K*np.exp(-r*T)) if side=="C" else (K*np.exp(-r*T) - S*np.exp(-q*T)))
    if target_price < intrinsic - 1e-5:
        return np.nan

    try:
        iv = brentq(price_error, 1e-6, 5.0) # Search for IV between 0.01% and 500%
    except (ValueError, RuntimeError):
        iv = np.nan
    return iv

# --- Vectorized MC Pricer ---
def price_heston_mc_vec(S_paths, K_vec, T, r, side='C', device=device):
    """Prices a vector of strikes from the same path tensor."""
    K_t = torch.tensor(K_vec, device=device).unsqueeze(0) # Shape [1, n_strikes]
    T_t = torch.tensor(T, device=device)
    r_t = torch.tensor(r, device=device)
    S_T = S_paths[:, -1].unsqueeze(1)

    if side == 'C':
        payoffs = torch.maximum(S_T - K_t, torch.tensor(0.0, device=device))
    else:
        payoffs = torch.maximum(K_t - S_T, torch.tensor(0.0, device=device))

    prices = torch.mean(payoffs, dim=0) * torch.exp(-r_t * T_t)
    return prices.cpu().numpy() # Return to CPU for IV calculation

# ===============================================================
# Step 2 — Prepare Calibration Target Data
# ===============================================================

# --- FIX 1: Select a "sweet spot" expiry, not a short-dated one ---
# 1. Clean the entire surface data first
iv_fit_clean = iv_fit_df.dropna(subset=['iv_fit'])
if iv_fit_clean.empty:
    raise RuntimeError("iv_fit_df contains no valid (non-NaN) IV data after interpolation.")

# 2. Find expiries in the "sweet spot" (3-9 months)
T_min, T_max = 0.25, 0.75
sweet_spot_expiries = iv_fit_clean[
    (iv_fit_clean['T'] > T_min) & (iv_fit_clean['T'] < T_max)
]['expiry'].unique()

if len(sweet_spot_expiries) == 0:
    print(f"Warning: No expiries found in sweet spot ({T_min}-{T_max} yrs). Using longest available.")
    # Fallback: find the longest-dated expiry that has data
    available_expiries = sorted(iv_fit_clean["T"].unique())
    if not available_expiries:
         raise RuntimeError("No valid expiry data found at all.")
    target_T = available_expiries[-1]
    target_expiry_date = iv_fit_clean[iv_fit_clean['T'] == target_T]['expiry'].iloc[0]
else:
    # Pick the first one in the sweet spot
    target_expiry_date = sweet_spot_expiries[0]
# --- End of Fix 1 ---


# 3. Select the target smile (it's guaranteed to have non-NaN data now)
target_smile = iv_fit_clean[iv_fit_clean["expiry"] == target_expiry_date].copy()
target_smile['K'] = target_smile['moneyness'] * S0

# These are our calibration targets
T_cal = target_smile['T'].iloc[0]
K_cal_grid = target_smile['K'].values
IV_cal_target = target_smile['iv_fit'].values

print(f"Calibrating to single expiry: {pd.to_datetime(target_expiry_date).date()} (T={T_cal:.4f})")
print(f"Target has {len(K_cal_grid)} smile points (after dropping NaNs).")

# ===============================================================
# Step 3 — Define The Objective (Loss) Function
# ===============================================================
iter_count = 0

def objective_func(params):
    """
    Loss function for the optimizer.
    Takes Heston params, returns sum-of-squared-IV-errors.
    """
    global iter_count
    iter_count += 1

    kappa, theta, sigma, rho, v0 = params

    # --- 1. Run MC Simulation (Smoother Version) ---
    try:
        S_paths_cal, _ = simulate_heston_paths(
            S0=S0, v0=v0, r=r, q=q,
            kappa=kappa, theta=theta, sigma=sigma, rho=rho,
            T=T_cal,
            n_steps=100, # Still fast steps
            n_paths=50_000, # <-- FIX 2: More paths for a smoother loss
            use_antithetic=True, seed=42, device=device
        )
    except (RuntimeError, ValueError):
        return 1e9 # Penalize unstable parameters

    # --- 2. Price the Smile ---
    P_mc = price_heston_mc_vec(S_paths_cal, K_cal_grid, T_cal, r, side='C')

    # --- 3. Convert Prices to IVs ---
    IV_mc = []
    for i, K in enumerate(K_cal_grid):
        iv = calculate_bs_iv(P_mc[i], S0, K, T_cal, r, q, side='C')

        if np.isnan(iv):
            if iter_count % 5 == 0:
                print(f"  Iter {iter_count}: Failed IV conversion (bad price). Penalizing.")
            return 1e9 # Return a massive error

        IV_mc.append(iv)
    IV_mc = np.array(IV_mc)

    # --- 4. Calculate Error ---
    error = np.sum((IV_mc - IV_cal_target)**2)

    if iter_count % 5 == 0:
        print(f"  Iter {iter_count}: Error={error:.6f} | Params (k,t,s,r,v0) = " \
              f"[{kappa:.3f}, {theta:.3f}, {sigma:.3f}, {rho:.3f}, {v0:.3f}]")

    if np.isnan(error):
        return 1e9

    return error

# ===============================================================
# Step 4 — Run the Optimizer
# ===============================================================

print("\nStarting Heston calibration... (This will take 10-20 minutes)")
print("Optimizer will print progress every 5 iterations.")

# Initial guess (our dummy parameters)
x0 = [2.0, 0.04, 0.3, -0.7, 0.04]

# Bounds: (kappa, theta, sigma, rho, v0)
bounds = [
    (0.1, 10.0),   # kappa (speed of reversion)
    (0.01, 1.0),   # theta (long-run variance)
    (0.01, 2.0),   # sigma (vol-of-vol)
    (-0.99, -0.1), # rho (leverage, usually negative)
    (0.01, 1.0)    # v0 (initial variance)
]

start_time = time.time()

# Run the minimization
result = minimize(
    objective_func,
    x0,
    method='L-BFGS-B',
    bounds=bounds,
    options={'maxiter': 150, 'eps': 1e-6, 'ftol': 1e-7} # <-- FIX 3: More iterations
)

end_time = time.time()
print(f"\nOptimization complete in {end_time - start_time:.2f} seconds.")

if result.success:
    print("✅ Calibration converged successfully.")
else:
    print(f"⚠️ Calibration did NOT converge: {result.message}")

best_params = result.x
kappa_fit, theta_fit, sigma_fit, rho_fit, v0_fit = best_params
print("\n--- Calibrated Heston Parameters ---")
print(f"  kappa (reversion): {kappa_fit:.4f}")
print(f"  theta (long-run var): {theta_fit:.4f} (vol = {np.sqrt(theta_fit):.2%})")
print(f"  sigma (vol-of-vol): {sigma_fit:.4f}")
print(f"  rho (correlation): {rho_fit:.4f}")
print(f"  v0 (initial var): {v0_fit:.4f} (vol = {np.sqrt(v0_fit):.2%})")

# ===============================================================
# Step 5 — Plot Final Model vs. Market
# ===============================================================

print("\nGenerating final high-quality plot...")

# Run ONE high-quality simulation with the *best* parameters
S_paths_final, _ = simulate_heston_paths(
    S0=S0, v0=v0_fit, r=r, q=q,
    kappa=kappa_fit, theta=theta_fit, sigma=sigma_fit, rho=rho_fit,
    T=T_cal, n_steps=252, n_paths=100_000, # <-- High quality
    use_antithetic=True, seed=42, device=device
)

# Price the smile
P_mc_final = price_heston_mc_vec(S_paths_final, K_cal_grid, T_cal, r, side='C')

# Convert prices to IVs
IV_mc_final = [calculate_bs_iv(P_mc_final[i], S0, K, T_cal, r, q, side='C') for i, K in enumerate(K_cal_grid)]

# Get the raw, non-smoothed market data for plotting
raw_market_data = df_greeks[
    (df_greeks['expiry'] == target_expiry_date) &
    (df_greeks['side'] == 'C') &
    (df_greeks['iv_clamped'].notna())
]

# --- Create the Plot ---
plt.figure(figsize=(10, 6))
# 1. Raw Market Data (what we're trying to fit)
plt.scatter(
    raw_market_data['strike'],
    raw_market_data['iv_clamped'],
    alpha=0.4, s=20, label=f"Market IVs (Raw)"
)
# 2. Smoothed Target (what we *actually* fit to)
plt.plot(
    K_cal_grid,
    IV_cal_target,
    'r--', linewidth=2, label="Smoothed Target Smile"
)
# 3. Heston Model (Our final result)
plt.plot(
    K_cal_grid,
    IV_mc_final,
    'g-x', linewidth=2, label="Calibrated Heston MC Smile"
)

plt.title(f"Heston Calibration vs. Market — {pd.to_datetime(target_expiry_date).date()}")
plt.xlabel("Strike (K)")
plt.ylabel("Implied Volatility")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig(f"figures_calibration/heston_fit_{pd.to_datetime(target_expiry_date).date()}.png", dpi=250)
plt.close()

print(f"✅ Cell 16 complete. Calibration plot saved to ./figures_calibration/")

Calibrating to single expiry: 2026-02-20 (T=0.3208)
Target has 120 smile points (after dropping NaNs).

Starting Heston calibration... (This will take 10-20 minutes)
Optimizer will print progress every 5 iterations.
  Iter 5: Error=1.022837 | Params (k,t,s,r,v0) = [2.000, 0.040, 0.300, -0.700, 0.040]
  Iter 10: Error=59.259205 | Params (k,t,s,r,v0) = [2.002, 1.000, 0.010, -0.990, 1.000]
  Iter 15: Error=1.903168 | Params (k,t,s,r,v0) = [2.000, 0.164, 0.263, -0.737, 0.164]
  Iter 20: Error=0.232528 | Params (k,t,s,r,v0) = [2.000, 0.082, 0.287, -0.713, 0.082]
  Iter 25: Error=0.230935 | Params (k,t,s,r,v0) = [2.000, 0.080, 0.288, -0.713, 0.082]
  Iter 30: Error=0.230935 | Params (k,t,s,r,v0) = [2.000, 0.080, 0.288, -0.713, 0.082]
  Iter 35: Error=0.230356 | Params (k,t,s,r,v0) = [2.000, 0.080, 0.289, -0.712, 0.082]
  Iter 40: Error=0.228091 | Params (k,t,s,r,v0) = [2.000, 0.077, 0.293, -0.711, 0.083]
  Iter 45: Error=0.196258 | Params (k,t,s,r,v0) = [1.998, 0.038, 0.346, -0.698, 0.097]
 